# Modélisation de la variabilité solaire à court terme

Dans ce notebook, nous cherchons à prédire la **variabilité future** de la production solaire normalisée.

La variable cible est définie par :
$$
y_t = |tch_{t+1} - tch_t|
$$

où `tch` représente la production solaire rapportée à la capacité maximale.

Comme chaque pas de temps correspond à **30 minutes**, cette cible mesure donc **l'amplitude de la variation de production dans les 30 minutes suivantes**.

L'objectif n'est donc pas de prévoir si la production va augmenter ou diminuer, mais plutôt **à quel point elle va changer**.

Nous allons progresser par étapes :

1. préparation des données de modélisation,
2. définition de métriques d'évaluation,
3. modèle naïf de référence,
4. modèles linéaires,
5. modèles d'arbres,
6. modèles de boosting,
7. réseau de neurones,
8. comparaison finale des performances.

In [ ]:
# ===========
# 1. Imports
# ===========
import numpy as np
import pandas as pd
from scipy.stats import uniform, randint, loguniform

import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from lightgbm import LGBMRegressor, plot_tree
from xgboost import XGBRegressor
from lazypredict.Supervised import LazyRegressor

import optuna
from optuna.pruners import MedianPruner

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import backend as K

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNetCV, ElasticNet
from sklearn.linear_model import Ridge, RidgeCV, Lasso, LassoCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNetCV, ElasticNet
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle
import joblib

## 1. Chargement des jeux de données

Nous utilisons ici les jeux de données déjà préparés dans le notebook de *feature engineering*.

Ces fichiers contiennent :
- la variable cible `target`,
- les variables retardées,
- les variables glissantes,
- les variables météo dérivées,
- les variables régionales agrégées.

In [ ]:
# ================================
# Chargement des données préparées
# ================================
filepath_input = "/Moustafa/data/local_data/04_Features_engineering/output/"
filepath_output = "/Moustafa/data/local_data/05_Modelisation/output/output_of_mi_modelisation_final_tuning_with_valid/"

train = pd.read_csv(filepath_input + "train_region.csv", index_col="datetime_utc", parse_dates=True)
valid = pd.read_csv(filepath_input + "valid_region.csv", index_col="datetime_utc", parse_dates=True)
test  = pd.read_csv(filepath_input + "test_region.csv",  index_col="datetime_utc", parse_dates=True)

print("Train :", train.shape)
display(train)

print("Valid :", valid.shape)
display(valid)

print("Test  :", test.shape)
display(test)


## 2. Séparation des variables explicatives et de la cible

La variable cible `target` représente :

$$
|tch_{t+1} - tch_t|
$$

Toutes les autres colonnes sont considérées comme variables explicatives disponibles à l'instant $t$.

In [ ]:
# ===================
# 3. Séparation X / y
# ===================
X_train = train.drop(columns=["target"])
y_train = train["target"]

X_valid = valid.drop(columns=["target"])
y_valid = valid["target"]

X_test = test.drop(columns=["target"])
y_test = test["target"]

print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("X_test  :", X_test.shape)

X_dev = pd.concat([X_train, X_valid]).sort_index()
y_dev = pd.concat([y_train, y_valid]).sort_index()

## 3. Création des folds temporels

Nous allons définir une fonction permettant de construire les *folds* utilisés pour l'entraînement et l'évaluation des modèles.  
Dans toute la suite, les modèles seront optimisés avec **Optuna**. La métrique de validation croisée reportée correspondra à la **MAE moyenne calculée sur les folds de validation**.

Comme il s'agit de **données temporelles**, les observations ne peuvent pas être mélangées aléatoirement. Le découpage doit respecter l'ordre chronologique : le modèle est toujours entraîné sur des données passées puis évalué sur des données futures.

On construit donc plusieurs couples *(train / validation)* qui se déplacent au fil du temps. Cette stratégie permet de reproduire des conditions proches de l'usage réel du modèle et de vérifier que ses performances restent robustes sur différentes périodes.



In [ ]:
#---------------------
# Folds temporels
#---------------------
def get_folds(X, y):
    """
    Génère des folds de validation croisée temporelle (time series CV).

    Principe :
    - On entraîne sur le passé
    - On valide sur le futur
    - On respecte un gap pour éviter toute fuite de données (data leakage)

    Paramètres
    ----------
    X : DataFrame indexé par date
    y : Series ou array aligné avec X

    Retour
    ------
    folds : liste de tuples (X_train, X_valid, y_train, y_valid)
    """

    folds = []
    
    # Définition manuelle des splits temporels
    #    2020 -> 2021
    #    2020-2021 -> 2022
    #    2020-2022 -> 2023
    #    2020-2023 -> 2024
    # Format: (date début train, date fin train, date début valid, date fin valid)
    splits = [
        ("2020-01-01", "2023-12-31", "2024-01-01", "2024-12-31"),
        ("2020-01-01", "2022-12-31", "2023-01-01", "2023-12-31"),
        ("2020-01-01", "2021-12-31", "2022-01-01", "2022-12-31"),
        ("2020-01-01", "2020-12-31", "2021-01-01", "2021-12-31"),
    ]

    for train_start, train_end, val_start, val_end in splits:

        # données train (passé)
        X_tr = X.loc[train_start:train_end]
        y_tr = y.loc[train_start:train_end]

        # données validation (futur)
        X_val = X.loc[val_start:val_end]
        y_val = y.loc[val_start:val_end]
        
        # sécurité temporelle (anti fuite)
        if X_tr.index.max() >= X_val.index.min():
            raise ValueError("Data leakage détecté : train chevauche validation")

        # ajout du fold
        folds.append((X_tr, X_val, y_tr, y_val))

    if len(folds) == 0:
        raise ValueError("Aucun fold valide généré. Vérifie les dates et l'index temporel.")

    return folds


## 4. Métriques d'évaluation

Pour comparer les performances des différents modèles, plusieurs métriques de régression sont utilisées.

**Métriques utilisées :**
- **MAE (Mean Absolute Error)** : erreur absolue moyenne  
- **NMAE (Normalized MAE)** : MAE normalisée (en %) pour faciliter la comparaison  
- **RMSE (Root Mean Squared Error)** : racine de l'erreur quadratique moyenne, pénalise davantage les erreurs importantes  
- **R² (coefficient de détermination)** : mesure la qualité de l'ajustement du modèle  
- **Best MAE CV** : meilleure MAE obtenue en validation croisée  

Dans ce notebook, les performances sont suivies sur les jeux :
- **train**,
- **validation**,
- **test**,

afin d'évaluer à la fois :
- la qualité d'ajustement,
- la capacité de généralisation,
- et le risque de surapprentissage.

Le MAE sera particulièrement utile ici car il mesure directement l'écart moyen sur la variabilité absolue prédite.

In [ ]:
# ============================================================
# 4. Fonction d'évaluation complète
# ============================================================
def calculer_nmae(y_true, y_pred, eps=1e-8):
    """
    Calcule la NMAE en pourcentage.

    Formule utilisée :
        NMAE = 100 * MAE / moyenne(|y_true|)

    eps permet d'éviter une division par zéro si la moyenne de y_true
    est extrêmement petite.
    """
    mae = mean_absolute_error(y_true, y_pred)
    denom = np.mean(np.abs(y_true)) + eps
    return 100 * mae / denom


def calculer_metriques(
    y_train, y_pred_train,
    y_valid, y_pred_valid,
    nom_modele,
    MAE_CV_moyenne=np.nan
):
    """
    Calcule les métriques de régression sur les jeux
    d'entraînement, de validation et de test.

    Paramètres
    ----------
    y_train, y_valid, y_test : array-like
        Valeurs réelles.
    y_pred_train, y_pred_valid, y_pred_test : array-like
        Prédictions du modèle.
    nom_modele : str
        Nom du modèle.
    MAE_CV_moyenne : float, optionnel
        Meilleure MAE obtenue en validation croisée.

    Retour
    ------
    dict
        Dictionnaire contenant toutes les métriques utiles.
    """

    # -------------------------
    # Métriques train
    # -------------------------
    mae_train = mean_absolute_error(y_train, y_pred_train)
    nmae_train = calculer_nmae(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)

    # -------------------------
    # Métriques validation
    # -------------------------
    mae_valid = mean_absolute_error(y_valid, y_pred_valid)
    nmae_valid = calculer_nmae(y_valid, y_pred_valid)
    rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
    r2_valid = r2_score(y_valid, y_pred_valid)


    return {
        "Modèle": nom_modele,

        "MAE_train": mae_train,
        "NMAE_train (%)": nmae_train,
        "RMSE_train": rmse_train,
        "R2_train": r2_train,

        "MAE_valid": mae_valid,
        "NMAE_valid (%)": nmae_valid,
        "RMSE_valid": rmse_valid,
        "R2_valid": r2_valid,

        "MAE_CV_moyenne": MAE_CV_moyenne,

        "Gap_MAE_valid_train": mae_valid - mae_train,
        "Gap_R2_train_valid": r2_train - r2_valid
    }


# Liste qui stockera les résultats finaux
resultats = []

# Baseline Modèles
## 1. Baseline naïve

Avant d'utiliser des modèles complexes, il est essentiel de définir une **référence simple**.

Comme la cible est une variation absolue future, une baseline naturelle consiste à supposer que la variabilité future sera proche de la variabilité observée récemment :

$$
\hat{y}_t = |\Delta tch_t|
$$

Dans nos variables explicatives, cela correspond à utiliser `abs_dtch_lag_1`.

In [ ]:
# ============================================================
# 1. Modèle naïf
# ============================================================
baseline_feature = "abs_dtch_dt"

y_pred_valid_naif = X_valid[baseline_feature]
y_pred_train_naif  = X_train[baseline_feature]
resultats.append(calculer_metriques(y_train, y_pred_train_naif, 
                                    y_valid, y_pred_valid_naif, 
                                    "Naïf (abs_dtch_lag_1)", MAE_CV_moyenne=np.nan))
pd.DataFrame(resultats)


## 2. Premier modèle linéaire inspiré de la physique

Une intuition simple est que de fortes variations de production solaire sont souvent liées à de fortes variations d'irradiance.

On peut donc commencer par un modèle linéaire très simple basé sur :

$$
|\Delta tch| \approx \beta_0 + \beta_1 |\Delta ghi|
$$

Dans nos variables, cela correspond à `region_abs_dghi_dt`.

In [ ]:
# ============================================================
# 2. Régression linéaire simple
# ============================================================
feature_physique = "region_abs_dghi_dt"
X_train_lr = X_train[[feature_physique]]
X_valid_lr = X_valid[[feature_physique]]

X_train_lrs = X_train_lr.iloc[1:]
X_valid_lrs = X_valid_lr.iloc[1:]
y_train_lrs = y_train[:-1]
y_valid_lrs = y_valid[:-1]

modele_lr_simple = LinearRegression()
modele_lr_simple.fit(X_train_lrs, y_train_lrs)

y_pred_valid_lrs = modele_lr_simple.predict(X_valid_lrs)
y_pred_train_lrs = modele_lr_simple.predict(X_train_lrs)

resultats.append(
    calculer_metriques(y_train_lrs, y_pred_train_lrs, 
                       y_valid_lrs, y_pred_valid_lrs, 
                       "Régression linéaire simple", MAE_CV_moyenne=np.nan))
pd.DataFrame(resultats)

## 3. Régression linéaire multivariée

La variabilité solaire ne dépend pas uniquement de l'irradiance. Elle peut aussi être influencée par :

- l'état récent de la production,
- les conditions de ciel clair,
- la nébulosité,
- les cycles temporels,
- la volatilité récente.

Nous enrichissons donc le modèle linéaire avec plusieurs variables explicatives.

In [ ]:
# ============================================================
# 3. Régression linéaire multivariée
# ============================================================
# features_lineaires = [
#     "region_dghi_dt",
#     "region_abs_dghi_dt",
#     "region_csi",
#     "region_nebulosite",
#     "abs_dtch_dt",
#     "abs_dtch_lag_1",
#     "abs_dtch_lag_2",
#     "tch_roll_std_4h",
#     "sin_hour",
#     "cos_hour",
#     "sin_doy",
#     "cos_doy"
# ]

# features_lineaires = [col for col in features_lineaires if col in X_train.columns]

# print("Variables retenues pour le modèle linéaire multivarié :")
# print(features_lineaires)
modele_lin_multi = LinearRegression()
modele_lin_multi.fit(X_train, y_train)

y_pred_valid_lin_multi = modele_lin_multi.predict(X_valid)
y_pred_train_lin_multi = modele_lin_multi.predict(X_train)

resultats.append(
    calculer_metriques(y_train, y_pred_train_lin_multi, 
                       y_valid, y_pred_valid_lin_multi, 
                       "Régression linéaire multivariée", 
                       MAE_CV_moyenne=np.nan))

pd.DataFrame(resultats)

## 4. Modèles linéaires régularisés : Ridge, Lasso et Elastic Net

Après la régression linéaire multivariée classique, il est naturel d'explorer des variantes **régularisées**.

En présence de nombreuses variables explicatives — notamment des variables retardées, glissantes et météorologiques — il peut exister :
- de la **colinéarité**,
- des coefficients instables,
- un risque de surajustement.

Nous testons donc trois approches complémentaires :

- **Ridge** : réduit progressivement l'amplitude des coefficients ;
- **Lasso** : peut annuler certains coefficients et donc effectuer une forme de sélection de variables ;
- **Elastic Net** : combine les avantages de Ridge et de Lasso, ce qui le rend particulièrement intéressant lorsque plusieurs variables sont corrélées entre elles.

Comme ces modèles pénalisent directement les coefficients, une **standardisation** des variables est appliquée avant l'ajustement.

### a. Ridge

In [ ]:
# Validation croisée temporelle
tscv = TimeSeriesSplit(n_splits=5)

# -----
# Ridge
# -----
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100], cv=tscv))
])

ridge.fit(X_train, y_train)

print("Meilleur alpha:", ridge.named_steps["model"].alpha_)

y_pred_train_ridge = ridge.predict(X_train)
y_pred_valid_ridge = ridge.predict(X_valid)

In [ ]:
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_ridge,
        y_valid, y_pred_valid_ridge,
        nom_modele="Ridge",
        MAE_CV_moyenne=np.nan
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df)

In [ ]:
coef = ridge.named_steps["model"].coef_
feature_names = X_train.columns
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coef
})
selected = coef_df[coef_df["coefficient"] != 0]
removed = coef_df[coef_df["coefficient"] == 0]

print("Selected features:")
display(selected.sort_values(by="coefficient", key=abs, ascending=False))

print("\nRemoved features:")
display(removed)

### b. Lasso

In [ ]:
# ------------------------------------------------------------
# Lasso
# ------------------------------------------------------------
lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(
        alphas=np.logspace(-3, 2, 50),
        cv=5,
        max_iter=50000
    ))
])

lasso.fit(X_train, y_train)

print("Meilleur alpha:", lasso.named_steps["model"].alpha_)

y_pred_train_lasso = lasso.predict(X_train)
y_pred_valid_lasso = lasso.predict(X_valid)

In [ ]:
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_lasso,
        y_valid, y_pred_valid_lasso,
        nom_modele="Lasso",
        MAE_CV_moyenne=np.nan
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df)

In [ ]:
# Récupérer les coefficients du modèle Lasso (après entraînement)
coef = lasso.named_steps["model"].coef_

# Créer un DataFrame associant chaque feature à son coefficient
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": coef
})

# Sélectionner les variables dont le coefficient est différent de 0 (features "gardées")
selected = coef_df.loc[coef_df["coefficient"] != 0]

# Sélectionner les variables dont le coefficient est égal à 0 (features "supprimées")
removed = coef_df.loc[coef_df["coefficient"] == 0]

# Afficher les features sélectionnées, triées par importance (valeur absolue du coefficient)
print("Features sélectionnées :")
display(selected.sort_values(by="coefficient", key=abs, ascending=False))

# Afficher les features supprimées
print("\nFeatures supprimées :")
display(removed)

### c. ElasticNet

In [ ]:
# ------------------------------------------
# Elastic Net avec recherche hyperparamètres 
# -------------------------------------------

elasticnet = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0],
        alphas=np.logspace(-3, 2, 50),
        cv=tscv,
        max_iter=50000,
        random_state=42,
        verbose=False
    ))
])

# Entraînement du modèle sur les données d'entraînement
elasticnet.fit(X_train, y_train)

# Prédictions sur le jeu d'entraînement
y_pred_train_enet = elasticnet.predict(X_train)

# Prédictions sur le jeu de validation
y_pred_valid_enet = elasticnet.predict(X_valid)

# Meilleurs hyperparamètres retenus
print("Meilleur alpha:", elasticnet.named_steps["model"].alpha_)
print("Meilleur l1_ratio:", elasticnet.named_steps["model"].l1_ratio_)

In [ ]:
# ---------------------
# Évaluation du modèle
# ---------------------
MAE_CV_moyenne_enet = np.nan

# Ajout des résultats dans une liste
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_enet,
        y_valid, y_pred_valid_enet,
        nom_modele="ElasticNetCV",
        MAE_CV_moyenne=MAE_CV_moyenne_enet
    )
)

# Affichage des résultats triés par performance (MAE validation)
pd.DataFrame(resultats).sort_values(by=["MAE_valid","Gap_MAE_valid_train"], ascending=[True, True])

In [ ]:
# Récupérer les coefficients du modèle ElasticNet (après entraînement)
coef = elasticnet.named_steps["model"].coef_

# Associer chaque feature à son coefficient dans un DataFrame
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": coef
})

# -------------------------------
# Sélection des variables
# -------------------------------

# Variables "gardées" : coefficient différent de 0
selected = coef_df.loc[coef_df["coefficient"] != 0]

# Variables "supprimées" : coefficient égal à 0
removed = coef_df.loc[coef_df["coefficient"] == 0]

# -------------------------------
# Affichage des résultats
# -------------------------------

# Afficher les features sélectionnées, triées par importance
# (importance = valeur absolue du coefficient)
print("Features sélectionnées :")
display(selected.sort_values(by="coefficient", key=abs, ascending=False))

# Afficher les features supprimées
print("\nFeatures supprimées :")
display(removed)

# Sélection automatique des meilleurs modèles avec LazyPredict

Après avoir testé des approches simples (modèle naïf, régression linéaire, etc...), l'objectif est maintenant d'explorer un large ensemble de modèles de machine learning afin d'identifier les plus performants.

Pour cela, on utilise **LazyPredict**, une librairie qui permet d'entraîner rapidement de nombreux modèles sans réglage manuel et de comparer leurs performances.

**Méthodologie:**

- Découpage des données en **train / validation** selon une logique temporelle
- Entraînement automatique de plusieurs modèles
- Évaluation des performances avec plusieurs métriques :
  - **MAE** (Mean Absolute Error)
  - **RMSE** (Root Mean Squared Error)
  - **R²** (coefficient de détermination)
- Calcul des métriques sur **train et validation** pour détecter l'overfitting
- Construction d'un **score global** pour classer les modèles

**Objectif:**

Identifier les **meilleurs modèles** et sélectionner les **5 plus performants**, qui seront ensuite analysés et optimisés plus finement.

In [ ]:

X_tr = X_train.loc[:"2022-12-31 23:30:00+00:00"]
y_tr = y_train.loc[:"2022-12-31 23:30:00+00:00"]

X_val = X_train.loc["2023-01-01 00:00:00+00:00":]
y_val = y_train.loc["2023-01-01 00:00:00+00:00":]

def run_lazy(X_train, X_valid, y_train, y_valid):
    """Fonction qui exécute LazyPredict sur un jeu d'entraînement et de validation
    et qui calcule :
    - MAE sur validation
    - MAE sur train (pour détecter l'overfitting)
    - NMAE: MAE normalisé (%) (pour train et validation)
    """

    # Initialisation de LazyRegressor
    reg = LazyRegressor(verbose=0, ignore_warnings=True, random_state=42, predictions=True)

    # Entraîne tous les modèles de LazyPredict
    # models : tableau contenant les métriques calculées par LazyPredict
    # preds : DataFrame contenant les prédictions de chaque modèle
    models, preds = reg.fit(X_train, X_valid, y_train, y_valid)

    # Moyennes des cibles (pour MAE normalisés)
    y_valid_mean = y_valid.mean()
    y_train_mean = y_train.mean()

    # Dictionnaires pour stocker les scores
    r2_scores_train = {}
    mae_val_scores = {}
    mae_rel_val_scores = {}
    mae_train_scores = {}
    mae_rel_train_scores = {}
    rmse_val_scores = {}
    rmse_rel_val_scores = {}
    rmse_train_scores = {}
    rmse_rel_train_scores = {}

    # Accès aux modèles entraînés
    trained_models = reg.models

    # Boucle sur chaque modèle
    for model_name in preds.columns:
        # ===== Validation =====
        pred_val = preds[model_name]

        # MAE validation
        mae_val = mean_absolute_error(y_valid, pred_val)

        # MAE relatif validation (%)
        mae_rel_val = round(mae_val / y_valid_mean * 100, 2) if y_valid_mean != 0 else np.nan

        # RMSE validation
        rmse_val = np.sqrt(mean_squared_error(y_valid, pred_val))

        # RMSE relatif validation (%)
        rmse_rel_val = round(rmse_val / y_valid_mean * 100, 2) if y_valid_mean != 0 else np.nan

        # ===== Train =====
        model = trained_models[model_name]

        # Prédictions sur train
        pred_train = model.predict(X_train)

        # R2 score train
        r2_score_train = r2_score(y_train, pred_train)
        
        # MAE train
        mae_train = mean_absolute_error(y_train, pred_train)

        # MAE relatif train (%)
        mae_rel_train = round(mae_train / y_train_mean * 100, 2) if y_train_mean != 0 else np.nan

        # RMSE train
        rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))

        # RMSE relatif train (%)
        rmse_rel_train = round(rmse_train / y_train_mean * 100, 2) if y_train_mean != 0 else np.nan

        # Stockage
        r2_scores_train[model_name] = r2_score_train
        mae_val_scores[model_name] = mae_val
        mae_rel_val_scores[model_name] = mae_rel_val
        mae_train_scores[model_name] = mae_train
        mae_rel_train_scores[model_name] = mae_rel_train
        rmse_val_scores[model_name] = rmse_val
        rmse_rel_val_scores[model_name] = rmse_rel_val
        rmse_train_scores[model_name] = rmse_train
        rmse_rel_train_scores[model_name] = rmse_rel_train

    # Création d'un DataFrame avec tous les scores
    mae_df = pd.DataFrame({
        "r2_score_train": pd.Series(r2_scores_train),
        "MAE_train": pd.Series(mae_train_scores),
        "MAE_relatif_train(%)": pd.Series(mae_rel_train_scores),
        "MAE_val": pd.Series(mae_val_scores),
        "MAE_relatif_val(%)": pd.Series(mae_rel_val_scores),
        "RMSE_train": pd.Series(rmse_train_scores),
        "RMSE_relatif_train(%)": pd.Series(rmse_rel_train_scores),
        "RMSE_val": pd.Series(rmse_val_scores),
        "RMSE_relatif_val(%)": pd.Series(rmse_rel_val_scores)
    })
    
    # Fusion avec les résultats LazyPredict
    out = models.merge(mae_df, left_index=True, right_index=True)

    return out, reg

# Exécuter LazyPredict
lazy_results_df, lazy_runner = run_lazy(X_tr, X_val, y_tr, y_val)

# Sauvegarder tous les modèles entraînés
joblib.dump(lazy_runner.models, "lazy_all_models.joblib")

# Renommer certaines colonnes pour avoir des noms plus explicites
lazy_results_df = lazy_results_df.rename({
    "R-Squared": "r2_val",
    "Time Taken": "fit_time_sec"
}, axis=1)

# Sélectionner uniquement les colonnes utiles pour l'analyse
lazy_metrics_df = lazy_results_df[
    [
        "r2_val", "r2_score_train",
        "MAE_train", "MAE_val",
        "MAE_relatif_train(%)", "MAE_relatif_val(%)",
        "RMSE_train", "RMSE_val",
        "RMSE_relatif_train(%)", "RMSE_relatif_val(%)",
        "fit_time_sec"
    ]
]

lazy_metrics_df = lazy_metrics_df.copy()

# Score d'overfitting
lazy_metrics_df["mae_gap"] = lazy_metrics_df["MAE_val"] - lazy_metrics_df["MAE_train"]
lazy_metrics_df["rmse_gap"] = lazy_metrics_df["RMSE_val"] - lazy_metrics_df["RMSE_train"]

# Score global (plus petit = meilleur)
lazy_metrics_df["global_score"] = (
    lazy_metrics_df["MAE_val"] * 0.7 +
    lazy_metrics_df["RMSE_val"] * 0.3
)

# Sauvegarde les résultats de LazyPredict dans un fichier CSV
# pour éviter de relancer l'entraînement (qui prend du temps)
lazy_metrics_df.reset_index(names="model_name").to_csv(
    "lazy_metrics.csv", index=False
)

# Top models
top_models_df = (
    lazy_metrics_df[lazy_metrics_df["r2_val"] > 0]
    .sort_values(["MAE_val", "r2_val"], ascending=[True, False])
    .head(40)
)

top_models_df

| Catégorie                          | Idée                                                         |
| ---------------------------------- | ------------------------------------------------------------ |
| **Modèles linéaires**              | supposent une relation linéaire entre variables              |
| **Machines à vecteurs de support (SVM)** | modèles non linéaires utilisant des noyaux                   |
| **Arbres de décision**             | règles de décision successives (tests sur les variables)     |
| **Ensembles d'arbres**             | combinaison de plusieurs arbres pour améliorer la prédiction (RandomForest, XGBoost…) |
| **Réseaux de neurones**            | modèles non linéaires (MLPRegressor)                   |
| **Méthodes à base de voisins**     | prédiction basée sur les k échantillons d'apprentissage les plus proches (selon une distance) du nouveau point (KNN)       |
| **Méthodes particulières**                   | modèles utilisant des stratégies particulières d'apprentissage ou de transformation pour améliorer la prédiction (TransformedTargetRegressor, PassiveAggressiveRegressor)         |


In [ ]:
model_categories = {

# Modèles linéaires
'LinearRegression': 'Modèles linéaires',
'Ridge': 'Modèles linéaires',
'RidgeCV': 'Modèles linéaires',
'Lasso': 'Modèles linéaires',
'LassoCV': 'Modèles linéaires',
'ElasticNet': 'Modèles linéaires',
'ElasticNetCV': 'Modèles linéaires',
'BayesianRidge': 'Modèles linéaires',
'HuberRegressor': 'Modèles linéaires',
'SGDRegressor': 'Modèles linéaires',
'LarsCV': 'Modèles linéaires',
'LassoLars': 'Modèles linéaires',
'LassoLarsCV': 'Modèles linéaires',
'LassoLarsIC': 'Modèles linéaires',
'OrthogonalMatchingPursuit': 'Modèles linéaires',
'OrthogonalMatchingPursuitCV': 'Modèles linéaires',
'PoissonRegressor': 'Modèles linéaires',
'TweedieRegressor': 'Modèles linéaires',

# Machines à vecteurs de support
'SVR': 'SVM',
'NuSVR': 'SVM',
'LinearSVR': 'SVM',

# Arbres de décision
'DecisionTreeRegressor': 'Arbres de décision',
'ExtraTreeRegressor': 'Arbres de décision',

# Ensembles d'arbres
'RandomForestRegressor': "Ensembles d'arbres",
'ExtraTreesRegressor': "Ensembles d'arbres",
'GradientBoostingRegressor': "Ensembles d'arbres",
'HistGradientBoostingRegressor': "Ensembles d'arbres",
'XGBRegressor': "Ensembles d'arbres",
'LGBMRegressor': "Ensembles d'arbres",
'BaggingRegressor': "Ensembles d'arbres",
'AdaBoostRegressor': "Ensembles d'arbres",

# Réseaux de neurones
'MLPRegressor': 'Réseaux de neurones',

# Méthodes basées sur les voisins
'KNeighborsRegressor': 'Méthodes à base de voisins',

# Méta-modèles
'TransformedTargetRegressor': 'Méthodes particulières',
'PassiveAggressiveRegressor': 'Méthodes particulières'
}

top_models_df['Category'] = top_models_df.index.map(model_categories)
top_models_df.head(10)

In [ ]:
category_performance = (
    top_models_df.groupby('Category')[['MAE_val', 'r2_val']]
    .agg(['mean','median','std','min','max']).sort_values(by=('MAE_val','mean'))
    .fillna('')
)

display(category_performance)

In [ ]:
# sélectionner la moyenne
scaled_plot = category_performance[[('MAE_val','mean'), ('r2_val','mean'), ('MAE_val','min'), ('r2_val','max')]]

# renommer les colonnes pour un affichage plus propre
scaled_plot.columns = ['MAE moyen', 'R² moyen', 'MAE min', 'R² max']

# graphique unscaled
scaled_plot.plot(kind='barh')
plt.title('Données scalées')
plt.xlabel('Moyenne R² / MAE')

plt.suptitle('Comparaison des performances des familles de modèles')

plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14,10))

sns.boxplot(x='Category',
            y='MAE_val',
            data=top_models_df,
            ax=axes[0])
axes[0].set_title("MAE Distribution by Model Family")
axes[0].tick_params(axis='x', rotation=90)
axes[0].set_ylim(0, 1.5)

sns.boxplot(x='Category',
            y='r2_val',
            data=top_models_df,
            ax=axes[1])
axes[1].set_title("R² Distribution by Model Family")
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

## 5. LightGBM

Nous commençons par **LightGBM**, un modèle de boosting particulièrement efficace sur les données tabulaires.

Ce modèle est intéressant ici car il peut :
- capturer des relations non linéaires,
- gérer des interactions complexes entre variables,
- rester relativement rapide à entraîner.

L'étude du modèle suit les étapes suivantes :

1. optimisation des hyperparamètres avec **Optuna**,
2. évaluation finale sur les jeux d'entraînement et de validation.

In [ ]:

def variables_importantes(model_class, X_train, y_train):
    """Calcule l'importance des variables pour un modèle donné."""

    # Instanciation du modèle
    model = model_class(random_state=42, n_jobs=-1)

    # Entraînement
    model.fit(X_train, y_train)

    # Récupération des importances
    importance = pd.Series(
        model.feature_importances_,
        index=X_train.columns
    ).sort_values(ascending=False)

    return importance


In [ ]:
# Importance des variables sur TRAIN uniquement
importance_lgbm = variables_importantes(LGBMRegressor, X_train, y_train)

# Calcul de l'importance cumulée normalisée
cum_importance_lgbm = importance_lgbm.cumsum() / importance_lgbm.sum()

# Sélection des variables expliquant jusqu'à 95% de l'importance
selected_features_lgbm = cum_importance_lgbm[cum_importance_lgbm <= 0.95].index

# Identification de la variable correspondant approximativement au seuil
index_threshold = cum_importance_lgbm[cum_importance_lgbm.round(2) == 0.95].index[0]

# Visualisation de l'importance cumulée des variables
plt.figure(figsize=(12,8))
plt.plot(cum_importance_lgbm.index, cum_importance_lgbm)
plt.axhline(0.95, color='red', linestyle='--')
plt.axvline(index_threshold, color='green', linestyle='--')
plt.xticks(rotation=90)
plt.ylabel("Cumulative importance")
plt.xlabel("Features")
plt.show()

# Visualisation de l'importance des variables sélectionnées
plt.figure(figsize=(8,12))
plt.barh(selected_features_lgbm, importance_lgbm[selected_features_lgbm]/importance_lgbm.sum())
plt.gca().invert_yaxis()
plt.title("Feature importance 'lgbm'")
plt.show()


### 10.2 Optimisation des hyperparamètres avec Optuna

L'optimisation du modèle LightGBM repose sur :

- une **validation croisée temporelle**,
- l'**early stopping**,
- une **pénalisation de l'overfitting** via l'écart train/validation,
- et le **pruning Optuna** pour accélérer la recherche.

L'objectif n'est pas seulement d'obtenir un modèle performant sur le train, mais surtout un modèle **robuste en validation**.

| Paramètre             | Explication                                                                                                                                                                                                        |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `n_estimators`        | Nombre d'arbres de boosting (itérations). Un plus grand nombre d'arbres améliore généralement les performances mais augmente le temps d'entraînement et le risque de surapprentissage (overfitting).               |
| `learning_rate`       | Taille du pas utilisée lors de la mise à jour des poids du modèle pendant le boosting. Des valeurs petites ralentissent l'apprentissage mais améliorent souvent la capacité de généralisation.                     |
| `num_leaves`          | Nombre maximal de feuilles par arbre. Des valeurs plus grandes permettent au modèle de capturer des relations plus complexes mais peuvent conduire au surapprentissage.                                            |
| `max_depth`           | Profondeur maximale de chaque arbre. `-1` signifie qu'il n'y a pas de limite. Limiter la profondeur aide à contrôler le surapprentissage et à réduire la complexité du modèle.                                     |
| `min_child_samples`   | Nombre minimum d'échantillons requis dans un nœud feuille. Des valeurs plus élevées rendent le modèle plus conservateur et réduisent le surapprentissage.                                                          |
| `subsample`           | Fraction des échantillons d'entraînement utilisée pour construire chaque arbre (échantillonnage des lignes). Des valeurs inférieures à 1 introduisent de l'aléatoire et améliorent la généralisation.              |
| `colsample_bytree`    | Fraction des variables (features) sélectionnées aléatoirement pour chaque arbre (échantillonnage des variables). Cela aide à réduire la corrélation entre les arbres et limite le surapprentissage.                |
| `reg_alpha`           | Terme de régularisation L1 appliqué aux poids des feuilles. Il favorise la parcimonie et réduit le surapprentissage en pénalisant les poids trop élevés.                                                           |
| `reg_lambda`          | Terme de régularisation L2 appliqué aux poids des feuilles. Il stabilise le modèle en pénalisant les coefficients trop grands et améliore la généralisation.                                                       |
| `estimator`           | Modèle d'apprentissage automatique à optimiser (ici c'est la régression LightGBM).                                                                                                                                              |
| `param_distributions` | Dictionnaire définissant l'espace de recherche des hyperparamètres.                                                                                                                                                |
| `n_iter`              | Nombre de combinaisons de paramètres testées aléatoirement. Des valeurs plus élevées explorent davantage de configurations mais augmentent le temps de calcul.                                                     |
| `scoring`             | Métrique d'évaluation utilisée pendant la validation croisée. `"neg_mean_absolute_error"` est utilisé car sklearn maximise les scores ; le signe négatif permet de transformer le MAE en problème de maximisation. |
| `cv`                  | Stratégie de validation croisée. Ici `TimeSeriesSplit` garantit que l'entraînement se fait sur des données passées et la validation sur des données futures.                                                       |
| `refit`               | Si `True`, le meilleur modèle trouvé pendant la validation croisée est réentraîné sur l'ensemble complet des données d'entraînement.                                                                               |


In [ ]:
X_train_lgbm = X_train.copy()
X_valid_lgbm = X_valid.copy()

#---------------------
# Évaluation des folds
#---------------------
def evaluate_params(params, X, y, trial=None):
    # listes pour stocker les métriques
    maes_valid = []; maes_train = []
    rmses_valid = []; rmses_train = []
    r2_trains = []; r2_valids = []
    gaps = []; best_iterations = []

    # génération des folds temporels
    folds = get_folds(X, y)

    # boucle sur les folds
    for step, (X_tr, X_v, y_tr, y_v) in enumerate(folds):

        model = LGBMRegressor(**params)

        # entraînement avec early stopping
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_v, y_v)],
            eval_metric="mae",
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )

        # récupérer best_iteration
        best_iter = model.best_iteration_ or params["n_estimators"]
        best_iterations.append(best_iter)
        
        # prédictions (meilleure itération)
        pred_train = model.predict(X_tr, num_iteration=best_iter)
        pred_valid = model.predict(X_v, num_iteration=best_iter)

        # calcul des métriques
        # MAE
        mae_train = mean_absolute_error(y_tr, pred_train)
        mae_valid = mean_absolute_error(y_v, pred_valid)
        
        # RMSE
        rmse_train = np.sqrt(mean_squared_error(y_tr, pred_train))
        rmse_valid = np.sqrt(mean_squared_error(y_v, pred_valid))
        
        # R2
        r2_train = r2_score(y_tr, pred_train)
        r2_valid = r2_score(y_v, pred_valid)

        # pénalité overfitting (gap train vs valid)
        gap = max(0, mae_valid - mae_train)

        # stockage
        maes_valid.append(mae_valid)
        maes_train.append(mae_train)
        rmses_valid.append(rmse_valid)
        rmses_train.append(rmse_train)
        r2_trains.append(r2_train)
        r2_valids.append(r2_valid)
        gaps.append(gap)

    # moyennes finales
    median_best_iteration = int(np.median(best_iterations))
    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)
    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)
    mean_r2_train = np.mean(r2_trains)
    mean_r2_valid = np.mean(r2_valids)
    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)
    mean_gap = np.mean(gaps)
    gap_pct = 100 * max(0, mean_mae_valid - mean_mae_train) / mean_mae_valid

    # sauvegarde des infos dans Optuna
    if trial is not None:
        trial.set_user_attr("best_iteration", median_best_iteration)

        trial.set_user_attr("mean_mae_valid", mean_mae_valid)
        trial.set_user_attr("mean_mae_train", mean_mae_train)

        trial.set_user_attr("std_mae_valid", std_mae_valid)
        trial.set_user_attr("std_mae_train", std_mae_train)
    
        trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
        trial.set_user_attr("mean_rmse_train", mean_rmse_train)
    
        trial.set_user_attr("mean_r2_valid", mean_r2_valid)
        trial.set_user_attr("mean_r2_train", mean_r2_train)
    
        trial.set_user_attr("mean_gap", mean_gap)
        trial.set_user_attr("std_gap", np.std(gaps))
        trial.set_user_attr("max_gap", np.max(gaps))
        trial.set_user_attr("overfit_gap_pct", gap_pct)
        
    # score final (à minimiser)
    return mean_mae_valid, mean_gap


#-------------------
# Fonction objective
#-------------------
def objective_lgbm(trial):

    # espace de recherche des hyperparamètres
    params = {
        "n_estimators": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
    
        "num_leaves": trial.suggest_int("num_leaves", 31, 128),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
    
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
    
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
    
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    
        "random_state": 42,
        "n_jobs": 1,
        "verbosity": -1
    }

    return evaluate_params(
        params,
        X_dev,
        y_dev,
        trial=trial
    )


#-------------------
# OPTUNA STUDY
#-------------------
study_lgbm = optuna.create_study(
    study_name="study_lgbm_new",
    storage=f"sqlite:///{filepath_output}study_lgbm_new.db",
    load_if_exists=True,
    directions=["minimize", "minimize"],
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=150,
        seed=42
    )
)

# lancement de l'optimisation
study_lgbm.optimize(
    objective_lgbm,
    n_trials=300,
    n_jobs=-1
)


In [ ]:
study_lgbm.trials_dataframe().sort_values(by=["values_0", "user_attrs_std_gap"])[['values_0', 'values_1', 'user_attrs_mean_mae_valid', 'user_attrs_mean_mae_train', "user_attrs_std_gap",'user_attrs_overfit_gap_pct']].head(10)

In [ ]:
X_train_lgbm = X_train.copy()
X_valid_lgbm = X_valid.copy()

study_lgbm = optuna.load_study(
    study_name="study_lgbm_new",
    storage=f"sqlite:///{filepath_output}study_lgbm_new.db"
)

#------------------
# TRAIN FINAL MODEL
#------------------
lgbm_trials = study_lgbm.best_trials

# choisir meilleur MAE avec gap faible
best_trial_lgbm = min(
    lgbm_trials,
    key=lambda t: t.values[0] + 0.5 * (t.values[1]/t.values[0])
)

print("Selected trial:")
print(f"MAE  = {best_trial_lgbm.values[0]}")
print(f"GAP  = {best_trial_lgbm.values[1]}")
# best_trial_lgbm = study_lgbm.trials[240]
best_params_lgbm = best_trial_lgbm.params
best_params_lgbm["n_estimators"] = best_trial_lgbm.user_attrs["best_iteration"]

best_model_lgbm = LGBMRegressor(
    verbosity=-1,
    random_state=42,
    n_jobs=-1,
    **best_params_lgbm
)

# entraînement final sur tout le train
best_model_lgbm.fit(X_train_lgbm, y_train)

#------------------
# EVALUATION
#------------------
# prédictions
y_pred_valid_lgbm = best_model_lgbm.predict(X_valid_lgbm)
y_pred_train_lgbm = best_model_lgbm.predict(X_train_lgbm)

# métriques
mae_valid_lgbm = mean_absolute_error(y_valid, y_pred_valid_lgbm)
mae_train_lgbm = mean_absolute_error(y_train, y_pred_train_lgbm)

rmse_valid_lgbm = np.sqrt(mean_squared_error(y_valid, y_pred_valid_lgbm))
rmse_train_lgbm = np.sqrt(mean_squared_error(y_train, y_pred_train_lgbm))

r2_valid_lgbm = r2_score(y_valid, y_pred_valid_lgbm)
r2_train_lgbm = r2_score(y_train, y_pred_train_lgbm)

nmae_valid_lgbm = round(mae_valid_lgbm / (y_valid.mean() + 1e-6) * 100, 3)
nmae_train_lgbm = round(mae_train_lgbm / (y_train.mean() + 1e-6) * 100, 3)

overfit_pct_lgbm = 100 * max(0, mae_valid_lgbm - mae_train_lgbm) / mae_valid_lgbm

#------------------------
# Résultats LGBMRegressor
#------------------------
print("\n--- Résultats LGBMRegressor ---")

print(f"MAE valid_lgbm     : {mae_valid_lgbm}")
print(f"MAE train_lgbm     : {mae_train_lgbm}")

print(f"NMAE valid_lgbm    : {nmae_valid_lgbm} %")
print(f"NMAE train_lgbm    : {nmae_train_lgbm} %")

print(f"RMSE valid_lgbm    : {rmse_valid_lgbm}")
print(f"RMSE train_lgbm    : {rmse_train_lgbm}")

print(f"R² valid_lgbm     : {r2_valid_lgbm}")
print(f"R² train_lgbm      : {r2_train_lgbm}")

print(f"Overfitting (CV)    : {best_trial_lgbm.user_attrs['overfit_gap_pct']:.2f} %")
print(f"Overfitting (final) : {overfit_pct_lgbm:.2f} %")

MAE_CV_moyenne_lgbm = best_trial_lgbm.user_attrs["mean_mae_valid"]

print("Best_params_lgbm:", best_trial_lgbm.params)
print("MAE_CV_moyenne_lgbm:", MAE_CV_moyenne_lgbm)

# -----------------------------
# Résumé standardisé des métriques
# -----------------------------
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_lgbm,
        y_valid, y_pred_valid_lgbm,
        nom_modele="LGBMRegressor",
        MAE_CV_moyenne=MAE_CV_moyenne_lgbm
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df)

In [ ]:
# resultats = [d for d in resultats if d["Modèle"] != "XGBRegressor"]
# resultats

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"
df_lgbm_study = study_lgbm.trials_dataframe()

fig = px.scatter(
    df_lgbm_study,
    x="user_attrs_mean_mae_valid",
    y="user_attrs_mean_gap",
    title="LGBM Trials: MAE vs Overfitting",
    hover_data=['number', 'values_0', 'values_1', 
                'user_attrs_max_gap', 'user_attrs_mean_gap', 
                'user_attrs_mean_mae_train', 'user_attrs_mean_mae_valid', 
                'user_attrs_mean_r2_train', 'user_attrs_mean_r2_valid', 
                'user_attrs_mean_rmse_train', 'user_attrs_mean_rmse_valid', 
                'user_attrs_overfit_gap_pct', 'user_attrs_std_gap', 'user_attrs_std_mae_train', 
                'user_attrs_std_mae_valid'],
)

fig.show()

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_valid, y_pred_valid_lgbm, alpha=0.5)
plt.plot([y_valid.min(), y_valid.max()],
         [y_valid.min(), y_valid.max()],
         'r--')
plt.xlabel("Valeurs réelles")
plt.ylabel("Valeurs prédites")
plt.title("LGBM - Réel vs Prédit pour valid")
plt.show()

## 6. XGBRegressor

In [ ]:
# Calcul et tri de l'importance des variables
importance_xgb = variables_importantes(XGBRegressor, X_train, y_train).sort_values(ascending=False)
threshold = 0.98

# Calcul de l'importance cumulée normalisée
cum_importance_xgb = importance_xgb.cumsum() / importance_xgb.sum()

# Sélection des variables expliquant jusqu'à 95% de l'importance
selected_features_xgb = cum_importance_xgb[cum_importance_xgb <= threshold].index

# Identification de la variable correspondant approximativement au seuil
index_threshold = cum_importance_xgb[cum_importance_xgb.round(2) == threshold].index[1]

# Visualisation de l'importance cumulée des variables
plt.figure(figsize=(12,8))
plt.plot(cum_importance_xgb.index, cum_importance_xgb)
plt.axhline(threshold, color='red', linestyle='--')
plt.axvline(index_threshold, color='green', linestyle='--')
plt.xticks(rotation=90)
plt.ylabel("Cumulative importance")
plt.xlabel("Features")
plt.show()

# Visualisation de l'importance des variables sélectionnées
plt.figure(figsize=(8,12))
plt.barh(selected_features_xgb, importance_xgb[selected_features_xgb]/importance_xgb.sum())
plt.gca().invert_yaxis()
plt.title("Feature importance 'xgbst'")
plt.show()

| Paramètre             | Explication                                                                                                                                                                           |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `n_estimators`        | Nombre d'arbres de boosting construits successivement. Une grande valeur peut améliorer les performances, mais augmente le temps de calcul et le risque de surapprentissage.          |
| `learning_rate`       | Taux d'apprentissage appliqué à la contribution de chaque arbre. Une petite valeur rend l'apprentissage plus progressif, mais nécessite souvent plus d'arbres.                        |
| `max_depth`           | Profondeur maximale de chaque arbre. Des arbres plus profonds capturent des relations plus complexes, mais peuvent surajuster les données.                                            |
| `min_child_weight`    | Poids minimal requis dans un nœud feuille pour autoriser une nouvelle division. Une valeur élevée rend le modèle plus conservateur.                                                   |
| `subsample`           | Fraction des observations d'entraînement utilisée pour construire chaque arbre. Une valeur inférieure à 1 peut aider à réduire le surapprentissage.                                   |
| `colsample_bytree`    | Fraction des variables utilisée pour construire chaque arbre. Cela permet de diversifier les arbres et de limiter le surapprentissage.                                                |
| `gamma`               | Gain minimal requis pour effectuer une nouvelle division dans un arbre. Plus `gamma` est grand, plus le modèle devient prudent dans les divisions.                                    |
| `reg_alpha`           | Régularisation L1 appliquée aux poids du modèle. Elle peut forcer certains poids à devenir nuls et ainsi rendre le modèle plus parcimonieux.                                          |
| `reg_lambda`          | Régularisation L2 appliquée aux poids du modèle. Elle pénalise les poids trop grands et aide à stabiliser le modèle.                                                                  |
| `tree_method`         | Méthode de construction des arbres. Ici `hist` utilise un algorithme basé sur des histogrammes, souvent plus rapide et efficace sur de grands jeux de données.                        |
| `max_bin`             | Nombre maximal d'intervalles utilisés pour discrétiser les variables continues avec `tree_method="hist"`. Une valeur plus grande peut améliorer la précision mais ralentir le calcul. |
| `n_iter`              | Nombre de combinaisons de paramètres testées aléatoirement parmi tout l'espace possible.                                                                                              |
| `scoring`             | Métrique utilisée pour comparer les modèles pendant la validation croisée. Ici `neg_mean_absolute_error` correspond à la MAE négative.                                                |
| `cv`                  | Méthode de validation croisée utilisée, ici `TimeSeriesSplit`, adaptée aux séries temporelles.                                                                                        |
| `refit`               | Si `True`, le meilleur modèle trouvé est réentraîné automatiquement sur toutes les données d'entraînement après la recherche.                                                         |

- If model overfits:
↑ gamma
↑ min_child_weight
↓ max_depth
↓ subsample
↑ reg_lambda
- If model underfits:
↓ gamma
↓ min_child_weight
↑ max_depth
↑ subsample
↓ regularization
- If learning is unstable:
↓ learning_rate
↑ n_estimators

### Optimisation XGBoost avec Optuna

- Validation croisée temporelle
- Early stopping
- Pénalisation de l'overfit
- Pruning Optuna

In [ ]:
X_train_xgb = X_train.copy()
X_valid_xgb = X_valid.copy()


#---------------------
# Évaluation des folds
#---------------------
def evaluate_params_xgb(params, X, y, trial=None):

    maes_valid = []; maes_train = []
    rmses_valid = []; rmses_train = []
    r2_trains = []; r2_valids = []
    gaps = []; best_iterations = []

    folds = get_folds(X, y)

    for step, (X_tr, X_v, y_tr, y_v) in enumerate(folds):

        model = XGBRegressor(
            **params,
            eval_metric="mae",
            early_stopping_rounds = 50
        )
        
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_v, y_v)],
            verbose=False
        )
        
        best_iter = model.best_iteration + 1
        best_iterations.append(best_iter)

        pred_train = model.predict(X_tr, iteration_range=(0, best_iter))
        pred_valid = model.predict(X_v, iteration_range=(0, best_iter))

        # métriques
        mae_train = mean_absolute_error(y_tr, pred_train)
        mae_valid = mean_absolute_error(y_v, pred_valid)

        rmse_train = np.sqrt(mean_squared_error(y_tr, pred_train))
        rmse_valid = np.sqrt(mean_squared_error(y_v, pred_valid))

        r2_train = r2_score(y_tr, pred_train)
        r2_valid = r2_score(y_v, pred_valid)

        gap = max(0, mae_valid - mae_train)

        maes_valid.append(mae_valid)
        maes_train.append(mae_train)
        rmses_valid.append(rmse_valid)
        rmses_train.append(rmse_train)
        r2_trains.append(r2_train)
        r2_valids.append(r2_valid)
        gaps.append(gap)

    median_best_iteration = int(np.median(best_iterations))

    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)
    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)

    mean_r2_train = np.mean(r2_trains)
    mean_r2_valid = np.mean(r2_valids)

    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)

    mean_gap = np.mean(gaps)
    gap_pct = 100 * max(0, mean_mae_valid - mean_mae_train) / mean_mae_valid

    
    if trial is not None:
        trial.set_user_attr("best_iteration", median_best_iteration)

        trial.set_user_attr("mean_mae_valid", mean_mae_valid)
        trial.set_user_attr("mean_mae_train", mean_mae_train)

        trial.set_user_attr("std_mae_valid", std_mae_valid)
        trial.set_user_attr("std_mae_train", std_mae_train)

        trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
        trial.set_user_attr("mean_rmse_train", mean_rmse_train)

        trial.set_user_attr("mean_r2_valid", mean_r2_valid)
        trial.set_user_attr("mean_r2_train", mean_r2_train)

        trial.set_user_attr("mean_gap", mean_gap)
        trial.set_user_attr("std_gap", np.std(gaps))
        trial.set_user_attr("max_gap", np.max(gaps))
        trial.set_user_attr("overfit_gap_pct", gap_pct)

    return mean_mae_valid, mean_gap

def objective_xgb(trial):

    max_depth = trial.suggest_int("max_depth", 3, 7)

    # max_depth + min_child_weight interaction
    if max_depth >= 7:
        min_child_weight = trial.suggest_int("min_child_weight", 10, 30)
    else:
        min_child_weight = trial.suggest_int("min_child_weight", 1, 30)

    params = {
        "n_estimators": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),

        "max_depth": max_depth,
        "min_child_weight": min_child_weight,

        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),

        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),

        "random_state": 42,
        "n_jobs": 1,
        "verbosity": 0
    }

    return evaluate_params_xgb(
        params,
        X_dev,
        y_dev,
        trial=trial
    )

study_xgb = optuna.create_study(
    study_name="study_xgb",
    storage=f"sqlite:///{filepath_output}study_xgb.db",
    load_if_exists=True,
    directions=["minimize", "minimize"],
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=100,
        seed=42
    ),
    pruner=MedianPruner(
        n_startup_trials=80,
        n_warmup_steps=2
    )
)

study_xgb.optimize(
    objective_xgb, 
    n_trials=200, 
    n_jobs=-1
)

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook"
df_xgb_study = study_xgb.trials_dataframe()

fig = px.scatter(
    df_xgb_study,
    x="user_attrs_mean_mae_valid",
    y="user_attrs_mean_gap",
    title="XGBoost Trials: MAE vs Overfitting",
    hover_data=['number', 'values_0', 'values_1', 
                'user_attrs_max_gap', 'user_attrs_mean_gap', 
                'user_attrs_mean_mae_train', 'user_attrs_mean_mae_valid', 
                'user_attrs_mean_r2_train', 'user_attrs_mean_r2_valid', 
                'user_attrs_mean_rmse_train', 'user_attrs_mean_rmse_valid', 
                'user_attrs_overfit_gap_pct', 'user_attrs_std_gap', 'user_attrs_std_mae_train', 
                'user_attrs_std_mae_valid'],
)

fig.show()

In [ ]:

X_train_xgb = X_train.copy()
X_valid_xgb = X_valid.copy()

study_xgb = optuna.load_study(
    study_name="study_xgb",
    storage=f"sqlite:///{filepath_output}study_xgb.db"
)

#------------------
# TRAIN FINAL MODEL
#------------------
xgb_trials = study_xgb.best_trials

best_trial_xgb = min(
    xgb_trials,
    key=lambda t: t.values[0] + 0.5 * (t.values[1]/t.values[0])
)
# best_trial_xgb = min(
#     pareto_trials,
#     key=lambda t: t.values[0] * (
#         1 + 0.3 * (t.user_attrs["overfit_gap_pct"] / 100)
#     )
# )

print("Selected trial:")
print(f"MAE  = {best_trial_xgb.values[0]}")
print(f"GAP  = {best_trial_xgb.values[1]}")
# best_trial_xgb = study_xgb.trials[198]

best_params_xgb = best_trial_xgb.params
best_params_xgb["n_estimators"] = best_trial_xgb.user_attrs["best_iteration"]

best_model_xgb = XGBRegressor(
    verbosity=0,
    random_state=42,
    n_jobs=-1,
    **best_params_xgb
)

best_model_xgb.fit(X_train_xgb, y_train)

y_pred_valid_xgb = best_model_xgb.predict(X_valid_xgb)
y_pred_train_xgb = best_model_xgb.predict(X_train_xgb)

mae_valid_xgb = mean_absolute_error(y_valid, y_pred_valid_xgb)
mae_train_xgb = mean_absolute_error(y_train, y_pred_train_xgb)

rmse_valid_xgb = np.sqrt(mean_squared_error(y_valid, y_pred_valid_xgb))
rmse_train_xgb = np.sqrt(mean_squared_error(y_train, y_pred_train_xgb))

r2_valid_xgb = r2_score(y_valid, y_pred_valid_xgb)
r2_train_xgb = r2_score(y_train, y_pred_train_xgb)

nmae_valid_xgb = round(mae_valid_xgb / y_valid.mean() * 100, 3)
nmae_train_xgb = round(mae_train_xgb / y_train.mean() * 100, 3)

overfit_pct_xgb = 100 * max(0, mae_valid_xgb - mae_train_xgb) / mae_valid_xgb

# ----------------
# Résultats
# ----------------
print("\n--- Résultats XGBRegressor ---")

print(f"MAE valid_xgb     : {mae_valid_xgb}")
print(f"MAE train_xgb     : {mae_train_xgb}")

print(f"NMAE valid_xgb    : {nmae_valid_xgb} %")
print(f"NMAE train_xgb    : {nmae_train_xgb} %")

print(f"RMSE valid_xgb    : {rmse_valid_xgb}")
print(f"RMSE train_xgb    : {rmse_train_xgb}")

print(f"R² valid_xgb      : {r2_valid_xgb}")
print(f"R² train_xgb      : {r2_train_xgb}")

print(f"Overfitting (CV)    : {best_trial_xgb.user_attrs['overfit_gap_pct']:.2f} %")
print(f"Overfitting (final) : {overfit_pct_xgb:.2f} %")
MAE_CV_moyenne_xgb = best_trial_xgb.user_attrs["mean_mae_valid"]

print("Best_params_xgb:", best_trial_xgb.params)
print("MAE_CV_moyenne_xgb:", MAE_CV_moyenne_xgb)

# -----------------------------
# Résumé standardisé des métriques
# -----------------------------
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_xgb,
        y_valid, y_pred_valid_xgb,
        nom_modele="XGBRegressor",
        MAE_CV_moyenne=MAE_CV_moyenne_xgb
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_train")
display(resultats_df)

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_valid, y_pred_valid_xgb, alpha=0.5)
plt.plot([y_valid.min(), y_valid.max()],
         [y_valid.min(), y_valid.max()],
         'r--')
plt.xlabel("Valeurs réelles")
plt.ylabel("Valeurs prédites")
plt.title("XGBoost - Réel vs Prédit")
plt.show()

## 7. ExtraTreesRegressor

In [ ]:
importance_etr = variables_importantes(ExtraTreesRegressor, X_train, y_train)

importance_etr = importance_etr.sort_values(ascending=False)

cum_importance_etr = importance_etr.cumsum() / importance_etr.sum()

selected_features_etr = cum_importance_etr[cum_importance_etr <= 0.95].index

plt.figure(figsize=(12,8))
plt.plot(cum_importance_etr.index, cum_importance_etr)
plt.axhline(0.95, color='red', linestyle='--')
plt.xticks(rotation=90)
plt.ylabel("Cumulative importance")
plt.xlabel("Features")
plt.title("Cumulative importance - ExtraTreesRegressor")
plt.show()

plt.figure(figsize=(8,12))
plt.barh(selected_features_etr, importance_etr[selected_features_etr] / importance_etr.sum())
plt.gca().invert_yaxis()
plt.title("Feature importance 'ExtraTreesRegressor'")
plt.show()

### Optimisation du modèle ExtraTreesRegressor


Dans ce partie d'optimisation avec Optuna, nous introduisons un **mode fast** afin d'accélérer la recherche d'hyperparamètres.

Le paramètre `fast_mode` permet de **réduire le coût des premières itérations**, en favorisant des évaluations plus rapides mais moins précises.

L'objectif est de **tester rapidement un grand nombre de configurations**, puis de **raffiner progressivement les meilleures solutions**.


**Étape 1 : Exploration** 

- Exploration d'un large espace d'hyperparamètres  
- Objectif principal : on veut tester beaucoup de configurations rapidement  
- Évaluations volontairement approximatives: les scores sont encore instables, pas besoin de calcul très précis

**But :** identifier les zones prometteuses de l'espace de recherche


**Étape 2 : Exploitation** 

- Concentration sur les meilleures zones identifiées par Optuna  
- Évaluations plus précises et plus robustes  
- Réduit le bruit (plus de folds CV): résultats plus stables  

**But :** affiner et valider les meilleurs hyperparamètres


In [ ]:
# ===========
# EXTRA TREES 
# ===========

X_train_etr = X_train.copy()
X_valid_etr = X_valid.copy()

# ---------------------
# Évaluation des folds
# ---------------------
def evaluate_params(params, X, y, trial=None):

    # stockage métriques
    maes_valid = []; maes_train = []
    rmses_valid = []; rmses_train = []
    r2_trains = []; r2_valids = []
    gaps = []
        
    folds = get_folds(X, y)

    for step, (X_tr, X_v, y_tr, y_v) in enumerate(folds):

        model = ExtraTreesRegressor(**params)

        # entraînement
        model.fit(X_tr, y_tr)

        # prédictions
        pred_train = model.predict(X_tr)
        pred_valid = model.predict(X_v)
        
        # MAE
        mae_train = mean_absolute_error(y_tr, pred_train)
        mae_valid = mean_absolute_error(y_v, pred_valid)
        
        # RMSE
        rmse_train = np.sqrt(mean_squared_error(y_tr, pred_train))
        rmse_valid = np.sqrt(mean_squared_error(y_v, pred_valid))
        
        # R2
        r2_train = r2_score(y_tr, pred_train)
        r2_valid = r2_score(y_v, pred_valid)
        
        # GAP
        gap = max(0, mae_valid - mae_train)
        
        # stockage
        maes_valid.append(mae_valid)
        maes_train.append(mae_train)
        rmses_valid.append(rmse_valid)
        rmses_train.append(rmse_train)
        r2_trains.append(r2_train)
        r2_valids.append(r2_valid)
        gaps.append(gap)

        # # pruning
        # if trial is not None:
        #     trial.report(np.mean(maes_valid), step)

        #     if trial.should_prune():
        #         raise optuna.exceptions.TrialPruned()

    # moyennes
    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)
    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)
    mean_r2_train = np.mean(r2_trains)
    mean_r2_valid = np.mean(r2_valids)
    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)
    mean_gap = np.mean(gaps)
    gap_pct = 100 * max(0, mean_mae_valid - mean_mae_train) / mean_mae_valid
    
    if trial is not None:

        trial.set_user_attr("mean_mae_valid", mean_mae_valid)
        trial.set_user_attr("mean_mae_train", mean_mae_train)

        trial.set_user_attr("std_mae_valid", std_mae_valid)
        trial.set_user_attr("std_mae_train", std_mae_train)
    
        trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
        trial.set_user_attr("mean_rmse_train", mean_rmse_train)
    
        trial.set_user_attr("mean_r2_valid", mean_r2_valid)
        trial.set_user_attr("mean_r2_train", mean_r2_train)
    
        trial.set_user_attr("mean_gap", mean_gap)
        trial.set_user_attr("std_gap", np.std(gaps))
        trial.set_user_attr("max_gap", np.max(gaps))
        trial.set_user_attr("overfit_gap_pct", gap_pct)
        
    # score final (à minimiser)
    return mean_mae_valid, mean_gap


# -------------------
# Fonction objective
# -------------------
def objective_etr(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 60),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 60),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
        "bootstrap": trial.suggest_categorical("bootstrap", [False, True]),
        "random_state": 42,
        "n_jobs": 1
    }

    return evaluate_params(
        params,
        X_dev,
        y_dev,
        trial=trial
    )


# -------------------
# OPTUNA STUDY
# -------------------
study_etr = optuna.create_study(
    study_name="study_etr",
    storage=f"sqlite:///{filepath_output}study_etr.db",
    load_if_exists=True,
    directions=["minimize", "minimize"],
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=60,
        seed=42
    )
)

study_etr.optimize(
    objective_etr,
    n_trials=300,
    n_jobs=-1
)

In [ ]:
study_etr.trials_dataframe().columns

In [ ]:
X_train_etr = X_train.copy()
X_valid_etr = X_valid.copy()

study_etr = optuna.load_study(
    study_name="study_etr",
    storage=f"sqlite:///{filepath_output}study_etr.db"
)
#------------------
# TRAIN FINAL MODEL
#------------------
etr_trials = study_etr.best_trials

best_trial_etr = min(
    etr_trials,
    key=lambda t: t.values[0] + 0.35 * (t.values[1]/t.values[0])
)
# best_trial_etr = min(
#     pareto_trials,
#     key=lambda t: t.values[0] * (
#         1 + 0.3 * (t.user_attrs["overfit_gap_pct"] / 100)
#     )
# )

print("Selected trial:")
print(f"MAE  = {best_trial_etr.values[0]}")
print(f"GAP  = {best_trial_etr.values[1]}")
# best_trial_etr = study_etr.trials[198]

best_params_etr = best_trial_etr.params


best_params_etr = {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 20, 
                   'min_samples_leaf': 30, 'max_features': 0.95, 'bootstrap': True}

best_model_etr = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1,
    **best_params_etr
)

# entraînement final sur tout le train
best_model_etr.fit(X_train_etr, y_train)

#------------------
# EVALUATION
#------------------
# prédictions
y_pred_valid_etr = best_model_etr.predict(X_valid_etr)
y_pred_train_etr = best_model_etr.predict(X_train_etr)

# métriques
mae_valid_etr = mean_absolute_error(y_valid, y_pred_valid_etr)
mae_train_etr = mean_absolute_error(y_train, y_pred_train_etr)

rmse_valid_etr = np.sqrt(mean_squared_error(y_valid, y_pred_valid_etr))
rmse_train_etr = np.sqrt(mean_squared_error(y_train, y_pred_train_etr))

r2_valid_etr = r2_score(y_valid, y_pred_valid_etr)
r2_train_etr = r2_score(y_train, y_pred_train_etr)

nmae_valid_etr = round(mae_valid_etr / (y_valid.mean() + 1e-6) * 100, 3)
nmae_train_etr = round(mae_train_etr / (y_train.mean() + 1e-6) * 100, 3)

overfit_pct_etr = 100 * max(0, mae_valid_etr - mae_train_etr) / mae_valid_etr

#------------------------
# Résultats LGBMRegressor
#------------------------
print("\n--- Résultats ExtraTreesRegressor ---")

print(f"MAE valid_etr     : {mae_valid_etr}")
print(f"MAE train_etr     : {mae_train_etr}")

print(f"NMAE valid_etr    : {nmae_valid_etr} %")
print(f"NMAE train_etr    : {nmae_train_etr} %")

print(f"RMSE valid_etr    : {rmse_valid_etr}")
print(f"RMSE train_etr    : {rmse_train_etr}")

print(f"R² valid_etr     : {r2_valid_etr}")
print(f"R² train_etr      : {r2_train_etr}")

print(f"Overfitting (CV)    : {best_trial_etr.user_attrs['overfit_gap_pct']} %")
print(f"Overfitting (final) : {overfit_pct_etr} %")

MAE_CV_moyenne_etr = best_trial_etr.user_attrs["mean_mae_valid"]

print("Best_params_etr:", best_trial_etr.params)
print("MAE_CV_moyenne_etr:", MAE_CV_moyenne_etr)

resultats = [r for r in resultats if r['Modèle']!='ExtraTreesRegressor']
# -----------------------------
# Résumé standardisé des métriques
# -----------------------------
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_etr,
        y_valid, y_pred_valid_etr,
        nom_modele="ExtraTreesRegressor",
        MAE_CV_moyenne=MAE_CV_moyenne_etr
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df)

In [ ]:
resultats = [r for r in resultats if r['Modèle']!='ExtraTreesRegressor']
# -----------------------------
# Résumé standardisé des métriques
# -----------------------------
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_etr,
        y_valid, y_pred_valid_etr,
        nom_modele="ExtraTreesRegressor",
        MAE_CV_moyenne=MAE_CV_moyenne_etr
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df)

### Conclusion

Le modèle `ExtraTreesRegressor` final présente un bon compromis entre performance et généralisation.

Après régularisation et optimisation des hyperparamètres avec Optuna :

- l'overfitting a été significativement réduit (écart train / validation plus faible),
- la performance en validation reste stable (R² ≈ 0.885),
- le modèle est désormais plus robuste et moins sensible aux variations des données.

Le modèle final est donc fiable pour la prédiction, avec une bonne capacité de généralisation sans sur-apprentissage excessif.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_valid, y_pred_valid_etr, alpha=0.5)
plt.plot([y_valid.min(), y_valid.max()],
         [y_valid.min(), y_valid.max()],
         'r--')
plt.xlabel("Valeurs réelles")
plt.ylabel("Valeurs prédites")
plt.title("ExtraTreesRegressor - Réel vs Prédit")
plt.show()

## 8. RandomForrestRegressor

In [ ]:

importance_rf = variables_importantes(RandomForestRegressor, X_train, y_train)

importance_rf = importance_rf.sort_values(ascending=False)

cum_importance_rf = importance_rf.cumsum() / importance_rf.sum()

selected_features_rf = cum_importance_rf[cum_importance_rf <= 0.95].index

# sécurité
if len(selected_features_rf) == 0:
    selected_features_rf = importance_rf.index[:1]


plt.figure(figsize=(12,8))
plt.plot(cum_importance_rf.index, cum_importance_rf)
plt.axhline(0.95, color='red', linestyle='--')
plt.xticks(rotation=90)
plt.ylabel("Cumulative importance")
plt.xlabel("Features")
plt.title("Cumulative importance - RandomForest")
plt.show()


plt.figure(figsize=(8,12))
plt.barh(
    selected_features_rf,
    importance_rf[selected_features_rf] / importance_rf.sum()
)
plt.gca().invert_yaxis()
plt.title("Feature importance 'RandomForest'")
plt.show()

### Optimisation RandomForrest avec Optuna

- Validation croisée temporelle
- Pénalisation de l'overfit
- Pruning Optuna

In [ ]:
# =============
# RANDOM FOREST
# =============

X_train_rf = X_train.copy()
X_valid_rf = X_valid.copy()

# ---------------------
# Évaluation des folds
# ---------------------
def evaluate_params(params, X, y, trial=None):

    # stockage métriques
    maes_valid = []; maes_train = []
    rmses_valid = []; rmses_train = []
    r2_trains = []; r2_valids = []
    gaps = []
        
    folds = get_folds(X, y)

    for step, (X_tr, X_v, y_tr, y_v) in enumerate(folds):

        model = RandomForestRegressor(**params)

        # entraînement
        model.fit(X_tr, y_tr)

        # prédictions
        pred_train = model.predict(X_tr)
        pred_valid = model.predict(X_v)
        
        # MAE
        mae_train = mean_absolute_error(y_tr, pred_train)
        mae_valid = mean_absolute_error(y_v, pred_valid)
        
        # RMSE
        rmse_train = np.sqrt(mean_squared_error(y_tr, pred_train))
        rmse_valid = np.sqrt(mean_squared_error(y_v, pred_valid))
        
        # R²
        r2_train = r2_score(y_tr, pred_train)
        r2_valid = r2_score(y_v, pred_valid)
        
        # GAP
        gap = max(0, mae_valid - mae_train)
        
        # stockage
        maes_valid.append(mae_valid)
        maes_train.append(mae_train)
        rmses_valid.append(rmse_valid)
        rmses_train.append(rmse_train)
        r2_trains.append(r2_train)
        r2_valids.append(r2_valid)
        gaps.append(gap)

        # # pruning
        # if trial is not None:
        #     trial.report(np.mean(maes_valid), step)

        #     if trial.should_prune():
        #         raise optuna.exceptions.TrialPruned()

    # moyennes
    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)
    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)
    mean_r2_train = np.mean(r2_trains)
    mean_r2_valid = np.mean(r2_valids)
    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)
    mean_gap = np.mean(gaps)
    gap_pct = 100 * max(0, mean_mae_valid - mean_mae_train) / mean_mae_valid
    
    if trial is not None:

        trial.set_user_attr("mean_mae_valid", mean_mae_valid)
        trial.set_user_attr("mean_mae_train", mean_mae_train)

        trial.set_user_attr("std_mae_valid", std_mae_valid)
        trial.set_user_attr("std_mae_train", std_mae_train)
    
        trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
        trial.set_user_attr("mean_rmse_train", mean_rmse_train)
    
        trial.set_user_attr("mean_r2_valid", mean_r2_valid)
        trial.set_user_attr("mean_r2_train", mean_r2_train)
    
        trial.set_user_attr("mean_gap", mean_gap)
        trial.set_user_attr("std_gap", np.std(gaps))
        trial.set_user_attr("max_gap", np.max(gaps))
        trial.set_user_attr("overfit_gap_pct", gap_pct)
        
    # score final (à minimiser)
    return mean_mae_valid, mean_gap


# -------------------
# Fonction objective
# -------------------
def objective_rf(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        "max_features": trial.suggest_float("max_features", 0.2, 0.7),
        "bootstrap": True,
        "max_samples": trial.suggest_float("max_samples", 0.5, 1.0),
        "random_state": 42,
        "n_jobs": 1
    }

    return evaluate_params(
        params,
        X_dev,
        y_dev,
        trial=trial
    )


# -------------------
# OPTUNA STUDY
# -------------------
study_rf = optuna.create_study(
    study_name="study_rf_new",
    storage=f"sqlite:///{filepath_output}study_rf_new.db",
    load_if_exists=True,
    directions=["minimize", "minimize"],
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=60,
        seed=42
    )
)

study_rf.optimize(
    objective_rf,
    n_trials=300,
    n_jobs=-1
)

In [ ]:
# X_train_rf = X_train.copy()
# X_valid_rf = X_valid.copy()

# study_rf = optuna.load_study(
#     study_name="study_rf_new",
#     storage=f"sqlite:///{filepath_output}study_rf_new.db"
# )

# ------------------
# TRAIN FINAL MODEL
# ------------------
rf_trials = study_rf.best_trials

# # choisir meilleur MAE avec gap faible
best_trial_rf = min(
    rf_trials,
    key=lambda t: t.values[0] + 0.5 * (t.values[1]/t.values[0])
)

print("Selected trial:")
print(f"MAE  = {best_trial_rf.values[0]}")
print(f"GAP  = {best_trial_rf.values[1]}")

# best_trial_rf = study_rf.trials[240]
best_params_rf = best_trial_rf.params
best_params_rf = {'n_estimators': 500, 
 'max_depth': 14, 'min_samples_split': 16, 
 'min_samples_leaf': 30, 'max_features': 0.2590161180162428, 
 'max_samples': 0.6465352183677964}
best_model_rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1,
    **best_params_rf
)

best_model_rf.fit(X_train_rf, y_train)


# ------------------
# EVALUATION
# ------------------
y_pred_valid_rf = best_model_rf.predict(X_valid_rf)
y_pred_train_rf = best_model_rf.predict(X_train_rf)

mae_valid_rf = mean_absolute_error(y_valid, y_pred_valid_rf)
mae_train_rf = mean_absolute_error(y_train, y_pred_train_rf)

rmse_valid_rf = np.sqrt(mean_squared_error(y_valid, y_pred_valid_rf))
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_pred_train_rf))

r2_valid_rf = r2_score(y_valid, y_pred_valid_rf)
r2_train_rf = r2_score(y_train, y_pred_train_rf)

nmae_valid_rf = round(mae_valid_rf / (y_valid.mean() + 1e-6) * 100, 3)
nmae_train_rf = round(mae_train_rf / (y_train.mean() + 1e-6) * 100, 3)

overfit_pct_rf = 100 * max(0, mae_valid_rf - mae_train_rf) / mae_valid_rf

# ----------------
# Résultats
# ----------------
print("\n--- Résultats RandomForrestRegressor ---")

print(f"MAE valid_rf     : {mae_valid_rf}")
print(f"MAE train_rf     : {mae_train_rf}")

print(f"NMAE valid_rf    : {nmae_valid_rf} %")
print(f"NMAE train_rf    : {nmae_train_rf} %")

print(f"RMSE valid_rf    : {rmse_valid_rf}")
print(f"RMSE train_rf    : {rmse_train_rf}")

print(f"R² valid_rf      : {r2_valid_rf}")
print(f"R² train_rf      : {r2_train_rf}")

print(f"Overfitting (CV)    : {best_trial_rf.user_attrs['overfit_gap_pct']} %")
print(f"Overfitting (final) : {overfit_pct_rf} %")

MAE_CV_moyenne_rf = best_trial_rf.user_attrs["mean_mae_valid"]

print("Best_params_rf:", best_trial_rf.params)
print("MAE_CV_moyenne_rf:", MAE_CV_moyenne_rf)

# -----------------------------
# Résumé standardisé des métriques
# -----------------------------
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_rf,
        y_valid, y_pred_valid_rf,
        nom_modele="RandomForrestRegressor",
        MAE_CV_moyenne=MAE_CV_moyenne_rf
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df)

## 9. HistGradientBoosting

### Optimisation HistGradient avec Optuna

- Validation croisée temporelle
- Pénalisation de l'overfit
- Pruning Optuna

In [ ]:
# ===============================
# HIST GRADIENT BOOSTING REGRESSOR
# ===============================

from sklearn.ensemble import HistGradientBoostingRegressor

X_train_hgb = X_train.copy()
X_valid_hgb = X_valid.copy()

# ---------------------
# Évaluation des folds
# ---------------------
def evaluate_params(params, X, y, trial=None):

    # stockage métriques
    maes_valid = []; maes_train = []
    rmses_valid = []; rmses_train = []
    r2_trains = []; r2_valids = []
    gaps = []
        
    folds = get_folds(X, y)

    for step, (X_tr, X_v, y_tr, y_v) in enumerate(folds):

        model = HistGradientBoostingRegressor(**params)

        # entraînement
        model.fit(X_tr, y_tr)

        # prédictions
        pred_train = model.predict(X_tr)
        pred_valid = model.predict(X_v)
        
        # MAE
        mae_train = mean_absolute_error(y_tr, pred_train)
        mae_valid = mean_absolute_error(y_v, pred_valid)
        
        # RMSE
        rmse_train = np.sqrt(mean_squared_error(y_tr, pred_train))
        rmse_valid = np.sqrt(mean_squared_error(y_v, pred_valid))
        
        # R²
        r2_train = r2_score(y_tr, pred_train)
        r2_valid = r2_score(y_v, pred_valid)
        
        # GAP (surapprentissage)
        gap = max(0, mae_valid - mae_train)
        
        # stockage
        maes_valid.append(mae_valid)
        maes_train.append(mae_train)
        rmses_valid.append(rmse_valid)
        rmses_train.append(rmse_train)
        r2_trains.append(r2_train)
        r2_valids.append(r2_valid)
        gaps.append(gap)

    # moyennes
    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)
    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)
    mean_r2_train = np.mean(r2_trains)
    mean_r2_valid = np.mean(r2_valids)
    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)
    mean_gap = np.mean(gaps)
    gap_pct = 100 * max(0, mean_mae_valid - mean_mae_train) / mean_mae_valid
    
    if trial is not None:

        trial.set_user_attr("mean_mae_valid", mean_mae_valid)
        trial.set_user_attr("mean_mae_train", mean_mae_train)

        trial.set_user_attr("std_mae_valid", std_mae_valid)
        trial.set_user_attr("std_mae_train", std_mae_train)
    
        trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
        trial.set_user_attr("mean_rmse_train", mean_rmse_train)
    
        trial.set_user_attr("mean_r2_valid", mean_r2_valid)
        trial.set_user_attr("mean_r2_train", mean_r2_train)
    
        trial.set_user_attr("mean_gap", mean_gap)
        trial.set_user_attr("std_gap", np.std(gaps))
        trial.set_user_attr("max_gap", np.max(gaps))
        trial.set_user_attr("overfit_gap_pct", gap_pct)
        
    # score final (multi-objective)
    return mean_mae_valid, mean_gap


# -------------------
# Fonction objective
# -------------------
def objective_hgb(trial):

    params = {
        "loss": "absolute_error",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        
        # nombre d'itérations (équivalent du nombre d'arbres)
        "max_iter": trial.suggest_int("max_iter", 100, 600),
        
        # complexité des arbres
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 12),

        # régularisation (éviter overfitting)
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 100),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.0, 5.0),
        "max_bins": trial.suggest_int("max_bins", 64, 255),
        
        "early_stopping": False,
        "random_state": 42,
        "verbose": 0
    }

    return evaluate_params(
        params,
        X_dev,
        y_dev,
        trial=trial
    )


# -------------------
# OPTUNA STUDY
# -------------------
study_hgb = optuna.create_study(
    study_name="study_hgb_new",
    storage=f"sqlite:///{filepath_output}study_hgb_new.db",
    load_if_exists=True,
    directions=["minimize", "minimize"],
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=60,
        seed=42
    )
)

study_hgb.optimize(
    objective_hgb,
    n_trials=150,
    n_jobs=-1
)

In [ ]:
# X_train_hgb = X_train.copy()
# X_valid_hgb = X_valid.copy()

# study_hgb = optuna.load_study(
#     study_name="study_hgb_new",
#     storage=f"sqlite:///{filepath_output}study_hgb_new.db"
# )

# ------------------
# TRAIN FINAL MODEL
# ------------------
rf_trials = study_hgb.best_trials

# # choisir meilleur MAE avec gap faible
best_trial_hgb = min(
    rf_trials,
    key=lambda t: t.values[0] + 0.5 * (t.values[1]/t.values[0])
)

print("Selected trial:")
print(f"MAE  = {best_trial_hgb.values[0]}")
print(f"GAP  = {best_trial_hgb.values[1]}")

# best_trial_hgb = study_hgb.trials[240]
best_params_hgb = best_trial_hgb.params

best_model_hgb = HistGradientBoostingRegressor(
    random_state=42,
    **best_params_hgb
)

best_model_hgb.fit(X_train_hgb, y_train)


# ------------------
# EVALUATION
# ------------------
y_pred_valid_hgb = best_model_hgb.predict(X_valid_hgb)
y_pred_train_hgb = best_model_hgb.predict(X_train_hgb)

mae_valid_hgb = mean_absolute_error(y_valid, y_pred_valid_hgb)
mae_train_hgb = mean_absolute_error(y_train, y_pred_train_hgb)

rmse_valid_hgb = np.sqrt(mean_squared_error(y_valid, y_pred_valid_hgb))
rmse_train_hgb = np.sqrt(mean_squared_error(y_train, y_pred_train_hgb))

r2_valid_hgb = r2_score(y_valid, y_pred_valid_hgb)
r2_train_hgb = r2_score(y_train, y_pred_train_hgb)

nmae_valid_hgb = round(mae_valid_hgb / (y_valid.mean() + 1e-6) * 100, 3)
nmae_train_hgb = round(mae_train_hgb / (y_train.mean() + 1e-6) * 100, 3)

overfit_pct_hgb = 100 * max(0, mae_valid_hgb - mae_train_hgb) / mae_valid_hgb

# ----------------
# Résultats
# ----------------
print("\n--- Résultats HistGradientBoostingRegressor ---")

print(f"MAE valid_hgb     : {mae_valid_hgb}")
print(f"MAE train_hgb     : {mae_train_hgb}")

print(f"NMAE valid_hgb    : {nmae_valid_hgb} %")
print(f"NMAE train_hgb    : {nmae_train_hgb} %")

print(f"RMSE valid_hgb    : {rmse_valid_hgb}")
print(f"RMSE train_hgb    : {rmse_train_hgb}")

print(f"R² valid_hgb      : {r2_valid_hgb}")
print(f"R² train_hgb      : {r2_train_hgb}")

print(f"Overfitting (CV)    : {best_trial_hgb.user_attrs['overfit_gap_pct']} %")
print(f"Overfitting (final) : {overfit_pct_hgb} %")

MAE_CV_moyenne_hgb = best_trial_hgb.user_attrs["mean_mae_valid"]

print("Best_params_hgb:", best_trial_hgb.params)
print("MAE_CV_moyenne_hgb:", MAE_CV_moyenne_hgb)

# -----------------------------
# Résumé standardisé des métriques
# -----------------------------
resultats = [res for res in resultats if res['Modèle']!='HistGradientBoostingRegressor']
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_hgb,
        y_valid, y_pred_valid_hgb,
        nom_modele="HistGradientBoostingRegressor",
        MAE_CV_moyenne=MAE_CV_moyenne_hgb
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df)

# 10. Deep Learning

### Optimisation MLPRegressor avec Optuna

- Validation croisée temporelle
- Pénalisation de l'overfit
- Pruning Optuna

In [ ]:
# ===========================
# FOLDS TEMPORELS
# ===========================
def get_folds(X, y):

    folds = []

    splits = [
        ("2020-01-01", "2023-12-31", "2024-01-01", "2024-12-31"),
        ("2020-01-01", "2022-12-31", "2023-01-01", "2023-12-31"),
        ("2020-01-01", "2021-12-31", "2022-01-01", "2022-12-31"),
        ("2020-01-01", "2020-12-31", "2021-01-01", "2021-12-31"),
    ]

    for train_start, train_end, val_start, val_end in splits:

        X_tr = X.loc[train_start:train_end]
        y_tr = y.loc[train_start:train_end]

        X_val = X.loc[val_start:val_end]
        y_val = y.loc[val_start:val_end]

        if X_tr.index.max() >= X_val.index.min():
            raise ValueError("Data leakage détecté")

        folds.append((X_tr, X_val, y_tr, y_val))

    return folds


# ===========================
# MODEL
# ===========================
def build_model(input_dim, params):

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(params["n_layers"]):
        model.add(layers.Dense(params["hidden_units"], activation=None))
        model.add(layers.BatchNormalization())
        model.add(layers.Activation("relu"))
        model.add(layers.Dropout(params["dropout"]))

    model.add(layers.Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["lr"]),
        loss="mae",
        metrics=["mae"]
    )

    return model


# ===========================
# OBJECTIVE MULTI
# ===========================
def objective(trial):

    params = {
        "n_layers": trial.suggest_int("n_layers", 1, 3),
        "hidden_units": trial.suggest_int("hidden_units", 64, 256),
        "dropout": trial.suggest_float("dropout", 0.0, 0.4),
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64]),
    }

    folds = get_folds(X_dev, y_dev)

    maes_train, maes_valid = [], []
    rmses_train, rmses_valid = [], []
    r2_trains, r2_valids = [], []
    gaps = []

    running_mae = []

    for step, (X_tr, X_val, y_tr, y_val) in enumerate(folds):

        model = build_model(X_tr.shape[1], params)

        callbacks = [
            keras.callbacks.EarlyStopping(
                patience=10,
                restore_best_weights=True,
                monitor="val_mae"
            ),
        ]

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=200,
            batch_size=params["batch_size"],
            verbose=0,
            callbacks=callbacks
        )

        pred_tr = model.predict(X_tr, verbose=0).flatten()
        pred_val = model.predict(X_val, verbose=0).flatten()

        # --- métriques ---
        mae_tr = np.mean(np.abs(y_tr - pred_tr))
        mae_val = np.mean(np.abs(y_val - pred_val))

        rmse_tr = np.sqrt(mean_squared_error(y_tr, pred_tr))
        rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))

        r2_tr = r2_score(y_tr, pred_tr)
        r2_val = r2_score(y_val, pred_val)

        gap = max(0, mae_val - mae_tr)

        # stockage
        maes_train.append(mae_tr)
        maes_valid.append(mae_val)

        rmses_train.append(rmse_tr)
        rmses_valid.append(rmse_val)

        r2_trains.append(r2_tr)
        r2_valids.append(r2_val)

        gaps.append(gap)

        # ===========================
        # PRUNING (sur MAE uniquement)
        # ===========================
        running_mae.append(mae_val)
        mean_so_far = np.mean(running_mae)

        trial.report(mean_so_far, step=step)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # ===========================
    # AGGREGATION
    # ===========================
    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)

    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)

    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)

    mean_r2_valid = np.mean(r2_valids)
    mean_r2_train = np.mean(r2_trains)

    mean_gap = np.mean(gaps)
    gap_pct = 100 * max(0, mean_mae_valid - mean_mae_train) / mean_mae_valid

    # ===========================
    # STORE
    # ===========================
    trial.set_user_attr("mean_mae_valid", mean_mae_valid)
    trial.set_user_attr("mean_mae_train", mean_mae_train)

    trial.set_user_attr("std_mae_valid", std_mae_valid)
    trial.set_user_attr("std_mae_train", std_mae_train)

    trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
    trial.set_user_attr("mean_rmse_train", mean_rmse_train)

    trial.set_user_attr("mean_r2_valid", mean_r2_valid)
    trial.set_user_attr("mean_r2_train", mean_r2_train)

    trial.set_user_attr("mean_gap", mean_gap)
    trial.set_user_attr("std_gap", np.std(gaps))
    trial.set_user_attr("max_gap", np.max(gaps))
    trial.set_user_attr("overfit_gap_pct", gap_pct)

    # ===========================
    # RETURN MULTI-OBJECTIVE
    # ===========================
    return mean_mae_valid, mean_gap


# ===========================
# STUDY MULTI
# ===========================
study = optuna.create_study(
    study_name="study_mlp_multi",
    storage=f"sqlite:///{filepath_output}study_mlp_multi.db",
    load_if_exists=True,
    directions=["minimize", "minimize"],
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    objective,
    n_trials=100,
    n_jobs=1,
    show_progress_bar=True
)


# ===========================
# RESULTATS
# ===========================
print("\nPareto front:")
for t in study.best_trials:
    print("params:", t.params)
    print("MAE:", t.values[0], "GAP:", t.values[1])
    print("-" * 40)

In [ ]:

# ---------------------
# Évaluation des folds
# ---------------------
def evaluate_params(params, X, y, trial=None):

    maes_valid = []; maes_train = []
    rmses_valid = []; rmses_train = []
    r2_trains = []; r2_valids = []
    gaps = []

    folds = get_folds(X, y)

    for step, (X_tr, X_v, y_tr, y_v) in enumerate(folds):

        # Pipeline pour MLP
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", MLPRegressor(**params))
        ])

        # entraînement
        model.fit(X_tr, y_tr)

        # prédictions
        pred_train = model.predict(X_tr)
        pred_valid = model.predict(X_v)

        # métriques
        mae_train = mean_absolute_error(y_tr, pred_train)
        mae_valid = mean_absolute_error(y_v, pred_valid)

        rmse_train = np.sqrt(mean_squared_error(y_tr, pred_train))
        rmse_valid = np.sqrt(mean_squared_error(y_v, pred_valid))

        r2_train = r2_score(y_tr, pred_train)
        r2_valid = r2_score(y_v, pred_valid)

        gap = max(0, mae_valid - mae_train)

        # stockage
        maes_valid.append(mae_valid)
        maes_train.append(mae_train)
        rmses_valid.append(rmse_valid)
        rmses_train.append(rmse_train)
        r2_trains.append(r2_train)
        r2_valids.append(r2_valid)
        gaps.append(gap)

    # moyennes
    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)
    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)
    mean_r2_train = np.mean(r2_trains)
    mean_r2_valid = np.mean(r2_valids)
    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)
    mean_gap = np.mean(gaps)
    gap_pct = 100 * max(0, mean_mae_valid - mean_mae_train) / mean_mae_valid

    if trial is not None:
        trial.set_user_attr("mean_mae_valid", mean_mae_valid)
        trial.set_user_attr("mean_mae_train", mean_mae_train)
        trial.set_user_attr("std_mae_valid", std_mae_valid)
        trial.set_user_attr("std_mae_train", std_mae_train)
        trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
        trial.set_user_attr("mean_rmse_train", mean_rmse_train)
        trial.set_user_attr("mean_r2_valid", mean_r2_valid)
        trial.set_user_attr("mean_r2_train", mean_r2_train)
        trial.set_user_attr("mean_gap", mean_gap)
        trial.set_user_attr("std_gap", np.std(gaps))
        trial.set_user_attr("max_gap", np.max(gaps))
        trial.set_user_attr("overfit_gap_pct", gap_pct)

    return mean_mae_valid, mean_gap


# -------------------
# Fonction objective
# -------------------
def objective_mlp(trial):
    n_layers = trial.suggest_int("n_layers", 1, 2)

    if n_layers == 1:
        hidden_layer_sizes = (
            trial.suggest_int("n_units_l1", 50, 200),
        )
    else:
        hidden_layer_sizes = (
            trial.suggest_int("n_units_l1", 50, 200),
            trial.suggest_int("n_units_l2", 50, 200),
        )
    params = {
        "hidden_layer_sizes": hidden_layer_sizes,
        "activation": trial.suggest_categorical(
            "activation",
            ["relu", "tanh"]
        ),
        "alpha": trial.suggest_float(
            "alpha", 1e-5, 1e-1, log=True
        ),
        "learning_rate_init": trial.suggest_float(
            "learning_rate_init", 1e-4, 1e-2, log=True
        ),
        "batch_size": trial.suggest_categorical(
            "batch_size", [32, 64, 128]
        ),
        "solver": trial.suggest_categorical("solver", ["adam"]),
        "max_iter": 1000,
        "early_stopping": True,
        "n_iter_no_change": 10,
        "random_state": 42
    }

    return evaluate_params(
        params,
        X_dev,
        y_dev,
        trial=trial
    )


# -------------------
# OPTUNA STUDY
# -------------------
study_mlp = optuna.create_study(
    study_name="study_mlp_new",
    storage=f"sqlite:///{filepath_output}study_mlp_new.db",
    load_if_exists=True,
    directions=["minimize", "minimize"],
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=60,
        seed=42
    )
)

study_mlp.optimize(
    objective_mlp,
    n_trials=300,
    n_jobs=-1
)

In [ ]:

# ===========================
# SEED (reproductibilité)
# ===========================
tf.random.set_seed(42)


# ===========================
# FOLDS TEMPORELS
# ===========================
def get_folds(X, y):
    """
    Génère des folds de validation croisée temporelle.

    Principe :
    - entraînement sur le passé
    - validation sur le futur
    - séparation temporelle explicite pour éviter le data leakage
    """

    folds = []

    splits = [
        ("2020-01-01", "2021-11-30", "2022-01-01", "2022-09-30"),
        ("2021-06-01", "2022-05-31", "2022-07-01", "2023-03-31"),
        ("2022-01-01", "2022-11-30", "2023-01-01", "2023-09-30"),
        ("2022-04-01", "2023-02-28", "2023-04-01", "2023-12-31")
    ]

    for train_start, train_end, val_start, val_end in splits:

        X_tr = X.loc[train_start:train_end]
        y_tr = y.loc[train_start:train_end]

        X_val = X.loc[val_start:val_end]
        y_val = y.loc[val_start:val_end]

        # sécurité : ignorer les folds vides
        if len(X_tr) == 0 or len(X_val) == 0:
            continue

        folds.append((X_tr, X_val, y_tr, y_val))

    return folds


# ===========================
# MODEL
# ===========================
def build_model(input_dim, params):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for _ in range(params["n_layers"]):
        model.add(
            layers.Dense(
                params["hidden_units"],
                activation="relu",
                kernel_initializer="he_normal"
            )
        )
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(params["dropout"]))

    model.add(layers.Dense(1))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["lr"]),
        loss="mae",
        metrics=["mae"]
    )

    return model


# ===========================
# EVALUATION CV
# ===========================
def evaluate_params_mlp(params, X, y, trial=None, fast_mode=True, alpha=0.7):

    maes_valid, maes_train = [], []
    rmses_valid, rmses_train = [], []
    r2_trains, r2_valids = [], []
    gaps = []

    folds = get_folds(X, y)

    # mode rapide : sous-échantillonnage de folds
    if fast_mode and len(folds) > 3:
        seed = trial.number if trial is not None else 42
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(folds), size=3, replace=False)
        folds = [folds[i] for i in idx]

    for step, (X_tr, X_v, y_tr, y_v) in enumerate(folds):

        K.clear_session()
        tf.random.set_seed(42)

        # ===========================
        # Scaling fold par fold
        # ===========================
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_v_scaled = scaler.transform(X_v)

        model = build_model(X_tr_scaled.shape[1], params)

        callbacks = [
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=3,
                min_lr=1e-5,
                verbose=0
            ),
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=8,
                restore_best_weights=True
            )
        ]

        model.fit(
            X_tr_scaled, y_tr,
            validation_data=(X_v_scaled, y_v),
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            verbose=0,
            callbacks=callbacks
        )

        pred_train = model.predict(X_tr_scaled, verbose=0).flatten()
        pred_valid = model.predict(X_v_scaled, verbose=0).flatten()

        mae_train = mean_absolute_error(y_tr, pred_train)
        mae_valid = mean_absolute_error(y_v, pred_valid)

        rmse_train = np.sqrt(mean_squared_error(y_tr, pred_train))
        rmse_valid = np.sqrt(mean_squared_error(y_v, pred_valid))

        r2_train = r2_score(y_tr, pred_train)
        r2_valid = r2_score(y_v, pred_valid)

        gap = max(0, mae_valid - mae_train)

        maes_train.append(mae_train)
        maes_valid.append(mae_valid)
        rmses_train.append(rmse_train)
        rmses_valid.append(rmse_valid)
        r2_trains.append(r2_train)
        r2_valids.append(r2_valid)
        gaps.append(gap)

        # pruning Optuna
        if trial is not None:
            score = np.mean(maes_valid) + alpha * np.mean(gaps)
            trial.report(score, step)

            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    # ===========================
    # Agrégation
    # ===========================
    mean_mae_valid = np.mean(maes_valid)
    mean_mae_train = np.mean(maes_train)

    std_mae_valid = np.std(maes_valid)
    std_mae_train = np.std(maes_train)

    mean_r2_train = np.mean(r2_trains)
    mean_r2_valid = np.mean(r2_valids)

    mean_rmse_valid = np.mean(rmses_valid)
    mean_rmse_train = np.mean(rmses_train)

    mean_gap = np.mean(gaps)

    final_score = mean_mae_valid + alpha * mean_gap

    if trial is not None:
        trial.set_user_attr("mean_mae_valid", mean_mae_valid)
        trial.set_user_attr("mean_mae_train", mean_mae_train)
        trial.set_user_attr("std_mae_valid", std_mae_valid)
        trial.set_user_attr("std_mae_train", std_mae_train)
        trial.set_user_attr("mean_rmse_valid", mean_rmse_valid)
        trial.set_user_attr("mean_rmse_train", mean_rmse_train)
        trial.set_user_attr("mean_r2_valid", mean_r2_valid)
        trial.set_user_attr("mean_r2_train", mean_r2_train)
        trial.set_user_attr("mean_gap", mean_gap)
        trial.set_user_attr("final_score", final_score)

    return final_score


# ===========================
# OBJECTIVE OPTUNA
# ===========================
def objective_mlp(trial):

    params = {
    "hidden_units": trial.suggest_categorical("hidden_units", [256]),
    "n_layers": trial.suggest_categorical("n_layers", [3]),
    "dropout": trial.suggest_float("dropout", 0.20, 0.40),
    "lr": trial.suggest_float("lr", 1e-3, 8e-3, log=True),
    "batch_size": trial.suggest_categorical("batch_size", [64]),
    "epochs": 50
    }

    return evaluate_params_mlp(
        params=params,
        X=X_full,
        y=y_full,
        trial=trial,
        fast_mode=False,
        alpha=0.7
    )


# ===========================
# OPTUNA
# ===========================
optuna.logging.set_verbosity(optuna.logging.INFO)

study_mlp = optuna.create_study(
    study_name="study_mlp",
    storage=f"sqlite:///{filepath_output}study_mlp.db",
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(n_startup_trials=10, seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)

study_mlp.optimize(
    objective_mlp,
    n_trials=25,
    n_jobs=1 
)


# ===========================
# MEILLEURS PARAMÈTRES
# ===========================
best_params_mlp = study_mlp.best_params.copy()
best_params_mlp["epochs"] = 50 


# ===========================
# SCALING FINAL
# ===========================
scaler_final = StandardScaler()

X_train_mlp = pd.DataFrame(
    scaler_final.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_valid_mlp = pd.DataFrame(
    scaler_final.transform(X_valid),
    columns=X_valid.columns,
    index=X_valid.index
)


# ===========================
# TRAIN FINAL MODEL
# ===========================
K.clear_session()
tf.random.set_seed(42)

best_model_mlp = build_model(X_train_mlp.shape[1], best_params_mlp)

history_mlp = best_model_mlp.fit(
    X_train_mlp.values,
    y_train,
    epochs=best_params_mlp["epochs"],
    batch_size=best_params_mlp["batch_size"],
    verbose=0
    )


# ===========================
# EVALUATION FINALE
# ===========================
y_pred_valid_mlp = best_model_mlp.predict(X_valid_mlp.values, verbose=0).flatten()
y_pred_train_mlp = best_model_mlp.predict(X_train_mlp.values, verbose=0).flatten()

mae_valid_mlp = mean_absolute_error(y_valid, y_pred_valid_mlp)
mae_train_mlp = mean_absolute_error(y_train, y_pred_train_mlp)

rmse_valid_mlp = np.sqrt(mean_squared_error(y_valid, y_pred_valid_mlp))
rmse_train_mlp = np.sqrt(mean_squared_error(y_train, y_pred_train_mlp))

r2_valid_mlp = r2_score(y_valid, y_pred_valid_mlp)
r2_train_mlp = r2_score(y_train, y_pred_train_mlp)

nmae_valid_mlp = round(mae_valid_mlp / (np.mean(np.abs(y_valid)) + 1e-6) * 100, 3)
nmae_train_mlp = round(mae_train_mlp / (np.mean(np.abs(y_train)) + 1e-6) * 100, 3)

print("\n--- Résultats Deep Learning (MLPRegressor) ---")
print(f"MAE valid_mlp     : {mae_valid_mlp:.6f}")
print(f"MAE train_mlp     : {mae_train_mlp:.6f}")

print(f"NMAE valid_mlp    : {nmae_valid_mlp} %")
print(f"NMAE train_mlp    : {nmae_train_mlp} %")

print(f"RMSE valid_mlp    : {rmse_valid_mlp:.6f}")
print(f"RMSE train_mlp    : {rmse_train_mlp:.6f}")

print(f"R² valid_mlp      : {r2_valid_mlp:.6f}")
print(f"R² train_mlp      : {r2_train_mlp:.6f}")

print("Best params      :", best_params_mlp)
print("Best MAE CV score:", study_mlp.best_trial.value)


In [ ]:
MAE_CV_moyenne_mlp = study_mlp.best_trial.user_attrs["mean_mae_valid"]

print("Best_params_mlp:", study_mlp.best_params)
print("MAE_CV_moyenne_mlp:", MAE_CV_moyenne_mlp)

# -----------------------------
# Résumé standardisé des métriques
# -----------------------------
resultats.append(
    calculer_metriques(
        y_train, y_pred_train_mlp,
        y_valid, y_pred_valid_mlp,
        nom_modele="",
        MAE_CV_moyenne=MAE_CV_moyenne_mlp
    )
)

resultats_df = pd.DataFrame(resultats).sort_values("MAE_valid")
display(resultats_df.set_index('Modèle'))

# Synthèse des performances des modèles

Ce tableau présente les principales métriques d'évaluation pour chaque modèle, calculées sur les jeux d'entraînement et de validation.

**Métriques utilisées :**
- **MAE (Mean Absolute Error)** : erreur absolue moyenne  
- **NMAE (Normalized MAE)** : MAE normalisée (en %) pour faciliter la comparaison  
- **RMSE (Root Mean Squared Error)** : pénalise davantage les erreurs importantes  
- **R² (coefficient de détermination)** : mesure la qualité de l'ajustement du modèle  
- **Best MAE CV** : meilleure MAE obtenue en validation croisée  

Cette synthèse permet de comparer rapidement les performances des modèles et d'identifier d'éventuels cas de surapprentissage (écart entre entraînement et validation).


In [ ]:
display(resultats_df)

In [ ]:
# =======================================================
# Visualisation des performances sur le jeu de validation
# =======================================================
resultats_valid = pd.DataFrame(resultats).sort_values("MAE_valid")

plt.figure(figsize=(10, 6))
plt.barh(resultats_valid["Modèle"], resultats_valid["MAE_valid"])
plt.xlabel("MAE_valid")
plt.ylabel("Modèle")
plt.title("Comparaison des modèles sur le jeu de validation")
plt.gca().invert_yaxis()
plt.show()